# SPACECONS — consensus of five spatial trajectory methods (CS690A Project 8)

Runs five spatial trajectory methods (**SpaceFlow, SpaTrack, StaVia, stLearn PSTS, CASCAT**) from one shared root and combines them:
**SPACECONS-R** (weighted rank consensus + bootstrap uncertainty), **SPACECONS-M** (consensus milestones and trajectory graph from each
method's own milestones) and **SPACECONS-3** (role-based consensus with middle fusion, section 10b). Every ordering is validated with
CASCAT's metrics (IM, OT, KT, SR, ARI, AMI), a state-edge F1 and a distance-from-root baseline.

**Datasets** (`DATASETS` in the download cell; run in this order):
* `'TBI'` — mouse brain 3 days after injury, Visium (GEO GSM7519182) — development
* `'BC'` — human breast cancer, Visium (10x Block A Section 1; SEDR annotation) — development
* `'ZHUANG'` — Zhuang-ABCA-2.005, MERFISH mouse brain (CASCAT figshare 28342754) — held-out test
* `'HER2ST'` — HER2+ breast cancer, ST (CASCAT figshare 28342754) — held-out test

The reference labels are used **only** to choose the root and to score — never to build an ordering, milestones or graph.

**How to run (Kaggle):** Settings → Accelerator **GPU T4**, Internet **on**, then **Run All** (about 2.5–3 h for all four datasets; stLearn
and CASCAT get their own Python environments, built with `uv`). Results: `/kaggle/working/v3_<DATASET>/`, the table
`/kaggle/working/validation_all_datasets.csv` and one zip per dataset (last cell). If figshare refuses downloads from Kaggle (HTTP 403),
download figshare files 52119332, 52119302 and 52119398 in a browser and add them as a Kaggle dataset (+ Add Input); the notebook finds
them by file size.


In [ ]:
# Main environment (Kaggle's own Python): SpaceFlow, SpaTrack, StaVia. 'dependency conflict' messages are expected and harmless.
# v3.1b: Kaggle's core packages (numpy, scipy, pandas, ...) are PINNED to the versions already installed, so pip cannot replace
# numpy underneath the running kernel (that caused "'numpy.ufunc' object has no attribute '__module__'").
import subprocess, sys
_keep = ('numpy', 'scipy', 'pandas', 'scikit-learn', 'numba', 'llvmlite', 'matplotlib', 'torch', 'anndata', 'h5py')
_frz = subprocess.run([sys.executable, '-m', 'pip', 'list', '--format=freeze'], capture_output=True, text=True).stdout.splitlines()
open('/tmp/pins.txt', 'w').write('\n'.join(l for l in _frz if l.split('==')[0].lower() in _keep) + '\n')
print('pinned:', open('/tmp/pins.txt').read().split())
!pip install -q -c /tmp/pins.txt --no-deps SpaceFlow spaTrack
!pip install -q -c /tmp/pins.txt torch-geometric gudhi cmcrameri scanpy igraph leidenalg pot pygam
!pip install -q -c /tmp/pins.txt --no-deps pyVIA pecanpy nptyping numba-progress s_gd2 graphtools tasklogger pygsp phate scprep
!pip install -q -c /tmp/pins.txt hnswlib datashader dask scikit-image "gensim>=4.4" deprecated future termcolor gdown wget umap-learn seaborn uv
import numpy; print('numpy', numpy.__version__, '(must equal the pinned version above)')


## Separate environments for stLearn and CASCAT
stLearn only installs on Python < 3.13 with NumPy < 2, and CASCAT's authors pin Python 3.10 + PyTorch 2.1.1 + PyG 2.6.1.
Kaggle's own Python is newer, so each gets its **own small environment** (made with `uv`). They run as separate programs,
read the same data, and hand back one pseudotime value per spot. This takes ~10 minutes the first time.

In [ ]:
# Two separate environments, made with `uv` (it downloads the right Python version itself):
#   stLearn needs Python < 3.13 and NumPy < 2        -> Python 3.11
#   CASCAT is pinned by its authors to Python 3.10, PyTorch 2.1.1 (CUDA 11.8) and PyG 2.6.1 -> exactly as in its README
import os, subprocess
ENVS = '/tmp/envs'; os.makedirs(ENVS, exist_ok=True)
STL_PY = f'{ENVS}/stl/bin/python'; CAS_PY = f'{ENVS}/cascat/bin/python'; CASCAT_DIR = f'{ENVS}/CASCAT'

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(('OK   ' if r.returncode == 0 else 'FAIL ') + cmd[:110])
    if r.returncode: print(r.stderr[-1500:])
    return r.returncode == 0

sh('pip install -q uv')
if not os.path.exists(STL_PY):
    sh(f'uv venv -q -p 3.11 {ENVS}/stl')
    sh(f'uv pip install -q -p {STL_PY} stlearn==1.2.2 "setuptools<81"')
if not os.path.exists(CAS_PY):
    sh(f'git clone -q --depth 1 https://github.com/deepomicslab/CASCAT {CASCAT_DIR}')
    sh(f'uv venv -q -p 3.10 {ENVS}/cascat')
    sh(f'uv pip install -q -p {CAS_PY} torch==2.1.1 --index-url https://download.pytorch.org/whl/cu118')
    sh(f'uv pip install -q -p {CAS_PY} torch_geometric==2.6.1 pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv '
       f'-f https://data.pyg.org/whl/torch-2.1.0+cu118.html')
    sh(f'uv pip install -q -p {CAS_PY} scanpy==1.10.1 matplotlib networkx scikit-misc pydot pot numpy==1.26.4 scikit-learn==1.5.2 '
       f'numba==0.60.0 numba-scipy==0.4.0 pandas==2.2.3 scipy==1.11.0 pyyaml==6.0.3 umap-learn seaborn cupy-cuda11x')   # cupy: GPU speed-up recommended in the README
sh(f'MPLBACKEND=Agg {STL_PY} -c "import stlearn, numpy; print(stlearn.__version__, numpy.__version__)"')
sh(f'MPLBACKEND=Agg {CAS_PY} -c "import torch, torch_sparse, torch_geometric; print(torch.__version__, torch.cuda.is_available())"')


## Download the data — choose the datasets in `DATASETS`
* `ZHUANG`, `HER2ST`: CASCAT's preprocessed files on figshare (article 28342754). Every file there is called `data.h5ad`, so the
  cell lists the record, downloads the most likely candidates and **checks** each (number of cells, annotation labels) before using it.
* `TBI`: GEO GSE236171 archive (~140 MB) → sample GSM7519182 arranged as a 10x Visium folder.
* `BC`: 10x Genomics *V1_Breast_Cancer_Block_A_Section_1* (via scanpy) + the per-spot pathology annotation (SEDR study).


In [ ]:
# [CELL:download]
# ===== choose the datasets (run in this order). Annotated CASCAT benchmarks: 'ZHUANG', 'HER2ST'; stLearn datasets: 'TBI', 'BC' =====
if 'DATASETS' not in globals():
    DATASETS = ['TBI', 'BC', 'ZHUANG', 'HER2ST']   # development (TBI, BC) first, then the held-out annotated datasets
ANNOTATED = ('ZHUANG', 'HER2ST')
DATASET = globals().get('NEXT_DATASET') or DATASETS[0]   # the 'next dataset' cell re-runs the notebook for the others
print('dataset:', DATASET, '| all datasets of this run:', DATASETS)
import os, glob, gzip, shutil, tarfile, subprocess
DATA_DIR = '/kaggle/working/data' if os.path.exists('/kaggle/working') else 'data'
os.makedirs(DATA_DIR, exist_ok=True)
VISIUM_OUT, H5AD_PATH, MILESTONE_CSV = None, None, None
if DATASET in ANNOTATED:
    # ---- CASCAT's figshare record: list its files, then download + check candidates until the right one is found ----
    import json as _json, urllib.request, anndata as _anndata, numpy as _np, pandas as pd
    FIG_API = 'https://api.figshare.com/v2/articles/28342754/files?page_size=200'
    files = _json.load(urllib.request.urlopen(urllib.request.Request(FIG_API, headers={'User-Agent': 'Mozilla/5.0'})))
    SPEC = {'ZHUANG': dict(first=52119332, size=14.9e6, n_obs=3161, key='0169'),        # first guess = file id, checked below
            'HER2ST': dict(first=52119398, size=49.3e6, n_obs=607,  key='gland')}[DATASET]
    cands = [f for f in files if f['name'].endswith('.h5ad')]
    cands.sort(key=lambda f: (f['id'] != SPEC['first'], abs(f['size'] - SPEC['size'])))
    UA = 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36'
    def _get(f):
        # v3.1e: 1) already downloaded  2) a copy you uploaded as a Kaggle dataset (same file size)  3) wget, then curl
        p = os.path.join(DATA_DIR, f"figshare_{f['id']}_{f['name']}")
        if os.path.exists(p): return p
        local = [q for q in glob.glob('/kaggle/input/**/*', recursive=True)
                 if os.path.isfile(q) and q.endswith(os.path.splitext(f['name'])[1]) and os.path.getsize(q) == f['size']]
        if local: print('  using uploaded copy', local[0]); return local[0]
        for cmd in (['wget', '-q', '-U', UA, '-O', p + '.part', f['download_url']],
                    ['curl', '-sSfL', '--retry', '3', '-A', UA, '-o', p + '.part', f['download_url']]):
            r = subprocess.run(cmd, capture_output=True, text=True)
            if r.returncode == 0 and os.path.exists(p + '.part') and os.path.getsize(p + '.part') > 0:
                os.rename(p + '.part', p); return p
            print(f"  {cmd[0]} failed for file {f['id']} (exit {r.returncode}) {(r.stderr or '').strip()[-150:]}")
        raise RuntimeError(f"download failed: {f['download_url']}")
    def _label_cols(A, key):                     # obs columns whose values contain the key word (e.g. '0169', 'gland')
        return [c for c in A.obs.columns if A.obs[c].astype(str).str.contains(key, case=False, regex=False).any()]
    for f in cands[:25]:
        try: p = _get(f)
        except RuntimeError as e_: print(' ', e_); continue
        try: A = _anndata.read_h5ad(p, backed='r')
        except Exception as e_: print('  skip', f['id'], repr(e_)[:80]); continue
        cols = _label_cols(A, SPEC['key'])
        if not cols and 'milestone_percentages' in A.uns and abs(A.n_obs - SPEC["n_obs"]) <= 0.35 * SPEC["n_obs"]:
            mp = pd.DataFrame(A.uns['milestone_percentages'])          # dynverse format: labels = milestone with the largest share
            lab_ = mp.sort_values('percentage').groupby('cell_id')['milestone_id'].last()
            if lab_.astype(str).str.contains(SPEC['key'], case=False, regex=False).any():
                A.file.close(); A = _anndata.read_h5ad(p); A.obs['cluster'] = lab_.reindex(A.obs_names.astype(str)).astype(str).values
                p = p.replace('.h5ad', '_with_cluster.h5ad'); A.write(p); A = _anndata.read_h5ad(p, backed='r'); cols = ['cluster']
        print(f"  checked figshare file {f['id']} ({f['size']/1e6:.1f} MB): {A.n_obs} x {A.n_vars}, label columns with "
              f"'{SPEC['key']}': {cols}")
        if cols and abs(A.n_obs - SPEC["n_obs"]) <= 0.35 * SPEC["n_obs"]:
            H5AD_PATH, LABEL_KEY = p, min(cols, key=lambda c: A.obs[c].nunique()); A.file.close(); break   # v3.1g: the label column, not a cell-ID column
        A.file.close()
    assert H5AD_PATH, (f'{DATASET}: no figshare file could be downloaded or matched (see above). If downloads fail, download '
                       f'https://figshare.com/articles/dataset/28342754 file {SPEC["first"]} in your browser and add it to this notebook '
                       f'as a Kaggle dataset (+ Add Input); it is then found automatically by its size.')
    if DATASET == 'ZHUANG':                      # the dataset's own milestone network (reference trajectory), if published
        for f in files:
            if f['name'] == 'milestone_network.csv':
                try: p = _get(f)
                except RuntimeError: print('milestone_network.csv not available -> the tree from the CASCAT paper text is used'); continue
                txt = open(p).read()
                if '0169' in txt: MILESTONE_CSV = p; print('milestone network file:', f['id']); print(txt[:400])
    print(f'{DATASET}: using {H5AD_PATH} | label column: {LABEL_KEY}')

elif DATASET == 'TBI':
    SAMPLE = 'GSM7519182'                  # TBI 3 dpi Visium (stLearn paper Fig. 2); GEO GSE236171
    VISIUM_OUT = os.path.join(DATA_DIR, SAMPLE)
    os.makedirs(DATA_DIR, exist_ok=True)

    tar_path = os.path.join(DATA_DIR, 'GSE236171_RAW.tar')
    if not glob.glob(f'{DATA_DIR}/**/{SAMPLE}*', recursive=True):
        if not os.path.exists(tar_path):
            subprocess.run(['wget', '-q', '-O', tar_path, 'https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE236171&format=file'], check=True)
        with tarfile.open(tar_path) as t: t.extractall(DATA_DIR)

    # GEO packs files per sample, often gzipped or as inner .tar(.gz): unpack everything once
    changed = True
    while changed:
        changed = False
        for f in glob.glob(f'{DATA_DIR}/**/*', recursive=True):
            if not os.path.isfile(f) or f == tar_path: continue
            if f.endswith(('.tar.gz', '.tgz', '.tar')):
                d = f.rsplit('.tar', 1)[0].rsplit('.tgz', 1)[0] + '_x'
                if not os.path.exists(d):
                    os.makedirs(d); tarfile.open(f).extractall(d); changed = True
            elif f.endswith('.gz') and not f.endswith(('matrix.mtx.gz', 'barcodes.tsv.gz', 'features.tsv.gz', 'genes.tsv.gz')):
                out = f[:-3]
                if not os.path.exists(out):
                    with gzip.open(f, 'rb') as a, open(out, 'wb') as b: shutil.copyfileobj(a, b)
                    changed = True

    files = sorted(f for f in glob.glob(f'{DATA_DIR}/**/*', recursive=True) if os.path.isfile(f) and f != tar_path)
    print('Files from GEO:'); [print('  ', os.path.relpath(f, DATA_DIR), f'({os.path.getsize(f)/1e6:.1f} MB)') for f in files]

    mine = [f for f in files if SAMPLE in f and not os.path.abspath(f).startswith(os.path.abspath(VISIUM_OUT) + os.sep)]   # v3.1c: skip our own output folder
    def pick(*keys, exclude=()):
        hits = [f for f in mine if all(k in os.path.basename(f).lower() for k in keys)
                and not any(e in os.path.basename(f).lower() for e in exclude) and not f.endswith('.gz')]
        return hits[0] if hits else None

    os.makedirs(f'{VISIUM_OUT}/spatial', exist_ok=True)
    h5 = pick('filtered', '.h5') or pick('.h5', exclude=('raw',))
    if h5:
        shutil.copy(h5, f'{VISIUM_OUT}/filtered_feature_bc_matrix.h5')
    else:                                             # matrix given as matrix.mtx / barcodes / features: rebuild the 10x .h5
        import scanpy as sc, h5py, numpy as np
        mtx = [f for f in mine if os.path.basename(f).lower().endswith(('matrix.mtx', 'matrix.mtx.gz'))][0]
        m = sc.read_10x_mtx(os.path.dirname(mtx), prefix=os.path.basename(mtx).split('matrix.mtx')[0], var_names='gene_symbols', make_unique=False)
        X = m.X.T.tocsc().astype(np.int32)
        with h5py.File(f'{VISIUM_OUT}/filtered_feature_bc_matrix.h5', 'w') as f:
            g = f.create_group('matrix'); g['barcodes'] = np.array(m.obs_names, dtype='S')
            g['data'], g['indices'], g['indptr'] = X.data, X.indices, X.indptr; g['shape'] = np.array(X.shape, dtype=np.int32)
            ft = g.create_group('features'); ft['name'] = np.array(m.var_names, dtype='S')
            ft['id'] = np.array(m.var['gene_ids'] if 'gene_ids' in m.var else m.var_names, dtype='S')
            ft['feature_type'] = np.array(['Gene Expression'] * m.n_vars, dtype='S'); ft['genome'] = np.array(['mm10'] * m.n_vars, dtype='S')
            ft['_all_tag_keys'] = np.array([b'genome'])
            f.attrs['filetype'] = 'matrix'; f.attrs['version'] = 2; f.attrs['library_ids'] = np.array([SAMPLE.encode()])
            f.attrs['chemistry_description'] = "Spatial 3' v1"; f.attrs['software_version'] = 'spaceranger-1.0.0'

    for keys, name in [(('tissue_positions',), 'tissue_positions_list.csv'), (('scalefactors',), 'scalefactors_json.json'),
                       (('hires',), 'tissue_hires_image.png'), (('lowres',), 'tissue_lowres_image.png')]:
        src = pick(*keys)
        if src and name == 'tissue_positions_list.csv':          # newer Space Ranger files have a header row; the classic format has none
            lines = open(src).read().splitlines()
            if lines and lines[0].lower().startswith('barcode'): lines = lines[1:]
            open(f'{VISIUM_OUT}/spatial/{name}', 'w').write('\n'.join(lines) + '\n')
        elif src: shutil.copy(src, f'{VISIUM_OUT}/spatial/{name}')
        else: print('MISSING for', SAMPLE, ':', keys)
    print('\nVisium folder ready:', VISIUM_OUT); print(sorted(os.listdir(f'{VISIUM_OUT}/spatial')))

else:
    import scanpy as sc
    SAMPLE = 'V1_Breast_Cancer_Block_A_Section_1'
    sc.settings.datasetdir = DATA_DIR
    sc.datasets.visium_sge(sample_id=SAMPLE)                # downloads the 10x matrix + spatial folder
    VISIUM_OUT = os.path.join(DATA_DIR, SAMPLE)
    ANNOT = os.path.join(DATA_DIR, 'BRCA1_metadata.tsv')     # pathology annotation per spot (SEDR study, JinmiaoChenLab)
    if not os.path.exists(ANNOT):
        subprocess.run(['wget', '-q', '-O', ANNOT,
                        'https://raw.githubusercontent.com/JinmiaoChenLab/SEDR_analyses/master/data/BRCA1/metadata.tsv'], check=True)
    print('Visium folder ready:', VISIUM_OUT, sorted(os.listdir(VISIUM_OUT)))
    print('annotation:', ANNOT)


## 0. Load the Visium data, QC as in the paper, preprocess
QC as in the stLearn paper: drop spots with < 200 detected genes. Then the same preprocessing as the rest of the project:
normalise → log → 2,000 highly variable genes → PCA (30) → neighbour graph → Leiden clusters (used as brain *regions*).

In [ ]:
# [CELL:load]
import glob, os, json
import numpy as np, pandas as pd, scanpy as sc, matplotlib.pyplot as plt
from scipy.stats import rankdata, spearmanr as _spearmanr
def spearmanr(a, b, **k):                     # ignores spots without a value (e.g. breast-cancer spots outside DCIS / IDC)
    return _spearmanr(np.asarray(a, float), np.asarray(b, float), nan_policy='omit')

OUT = f'/kaggle/working/v3_{DATASET}' if os.path.exists('/kaggle/working') else f'outputs_v3_{DATASET}'
os.makedirs(OUT, exist_ok=True)

if DATASET in ANNOTATED:
    # ---- CASCAT benchmark file: counts + coordinates + curated labels in one h5ad ----
    VISIUM_DIR = None
    adata = sc.read_h5ad(H5AD_PATH)
    adata.obs_names = adata.obs_names.astype(str); adata.obs_names_make_unique(); adata.var_names_make_unique()
    if 'counts' in adata.layers: adata.X = adata.layers['counts']
    sp_key = next((k for k in ['spatial', 'X_spatial', 'spatial_coords'] if k in adata.obsm), None)
    if sp_key: coords = np.asarray(adata.obsm[sp_key], float)[:, :2]
    else:
        pair = next(((a, b) for a, b in [('x', 'y'), ('X', 'Y'), ('center_x', 'center_y'), ('x_section', 'y_section'),
                                          ('imagecol', 'imagerow'), ('array_col', 'array_row')]
                     if a in adata.obs and b in adata.obs), None)
        assert pair, f'no spatial coordinates found; obsm: {list(adata.obsm)}, obs: {list(adata.obs.columns)}'
        coords = adata.obs[list(pair)].to_numpy(float)
    adata.obsm['spatial'] = coords
    adata.uns = {k: v for k, v in adata.uns.items() if k in ('milestone_network', 'spatial')}
    Xd = adata.X[:200].toarray() if hasattr(adata.X, 'toarray') else np.asarray(adata.X[:200])
    LOGGED = bool(np.nanmax(Xd) < 30 and not np.allclose(Xd, np.round(Xd)))       # already log-normalised?
    print('raw:', adata.shape, '| obs columns:', list(adata.obs.columns)[:12], '| obsm:', list(adata.obsm),
          '| X max', round(float(np.nanmax(Xd)), 2), '| already log-normalised:', LOGGED)
    print('labels (', LABEL_KEY, '):', adata.obs[LABEL_KEY].astype(str).value_counts().head(12).to_dict())
    sc.pp.filter_cells(adata, min_counts=1)          # curated benchmark: keep every cell with any signal (MERFISH panels are small)
else:
  # ---- find the 10x Visium folder (the one that contains filtered_feature_bc_matrix.h5 and spatial/) ----
  LOGGED = False
  cands = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/filtered_feature_bc_matrix.h5', recursive=True)
         + glob.glob('**/filtered_feature_bc_matrix.h5', recursive=True)]
  cands = [c for c in cands if os.path.exists(os.path.join(c, 'spatial', 'tissue_positions_list.csv'))
           or os.path.exists(os.path.join(c, 'spatial', 'tissue_positions.csv'))]
  if not cands and not os.path.exists(f'{VISIUM_OUT}/filtered_feature_bc_matrix.h5'):
      raise FileNotFoundError('Add the Visium folder as a Kaggle input (it must contain filtered_feature_bc_matrix.h5 and spatial/)')
  VISIUM_DIR = VISIUM_OUT if os.path.exists(f'{VISIUM_OUT}/filtered_feature_bc_matrix.h5') else cands[0]
  print('Visium folder:', VISIUM_DIR)
  adata = sc.read_visium(VISIUM_DIR); adata.var_names_make_unique()
  print('raw:', adata.shape)
  sc.pp.filter_cells(adata, min_genes=200)         # QC exactly as in the stLearn paper (Visium)

def rank01(t):
    t = np.asarray(t, dtype=float)
    t = np.where(np.isfinite(t), t, np.nanmax(t[np.isfinite(t)]))
    return (rankdata(t) - 1) / (len(t) - 1)

# ---- genes seen in < 3 spots are dropped (all datasets) ----
sc.pp.filter_genes(adata, min_cells=3)
raw = adata.copy()                                   # raw counts (SpaceFlow / stLearn / CASCAT normalise themselves)
ad = adata.copy()
if not LOGGED:
    sc.pp.normalize_total(ad, target_sum=1e4); sc.pp.log1p(ad)
N_HVG = min(2000, ad.n_vars)
if ad.n_vars <= 2000: ad.var['highly_variable'] = True          # small targeted panel (MERFISH): use every gene
else: sc.pp.highly_variable_genes(ad, n_top_genes=N_HVG)
sc.tl.pca(ad, n_comps=min(30, ad.n_vars - 1), mask_var='highly_variable', random_state=0)
sc.pp.neighbors(ad, n_neighbors=15, n_pcs=30, random_state=0)
sc.tl.leiden(ad, resolution=0.8, key_added='region', flavor='igraph', n_iterations=2, directed=False, random_state=0)
xy = np.asarray(ad.obsm['spatial'], dtype=float)[:, :2]
open(f'{OUT}/kept_spots.txt', 'w').write('\n'.join(ad.obs_names))
print('after QC:', ad.shape, '| expression clusters (regions):', ad.obs['region'].nunique())

# ---- tissue image helpers (Visium: H&E image; MERFISH / HER2ST h5ad: plain coordinates, no image) ----
def tissue_image():
    S = next(iter(ad.uns.get('spatial', {}).values()), None) if isinstance(ad.uns.get('spatial'), dict) else None
    if S is not None and 'images' in S and 'hires' in S['images'] and 'scalefactors' in S:
        return S['images']['hires'], xy * S['scalefactors']['tissue_hires_scalef']
    return None, xy.copy()
TISSUE_IMG, _ = tissue_image()
def show_tissue(ax, alpha=1.0):
    if TISSUE_IMG is not None: ax.imshow(TISSUE_IMG, alpha=alpha)
    else:
        ax.set_aspect('equal')
        if not ax.yaxis_inverted(): ax.invert_yaxis()

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
if TISSUE_IMG is not None:
    sc.pl.spatial(ad, color=None, ax=axes[0], show=False, title='H&E image')
    sc.pl.spatial(ad, color='region', ax=axes[1], show=False, title='Expression clusters (Leiden)', spot_size=None)
else:
    lab0 = ad.obs[LABEL_KEY].astype(str).values
    for ax_, lab_, ttl in [(axes[0], lab0, f'annotation ({LABEL_KEY})'), (axes[1], ad.obs['region'].astype(str).values, 'Expression clusters (Leiden)')]:
        show_tissue(ax_)
        for k_ in pd.unique(lab_): m_ = lab_ == k_; ax_.scatter(xy[m_, 0], xy[m_, 1], s=4, label=str(k_)[:28])
        ax_.set_title(ttl); ax_.axis('off')
    axes[1].legend(fontsize=7, markerscale=3, ncol=2, loc='center left', bbox_to_anchor=(1, .5))
plt.tight_layout(); plt.show()


In [ ]:
# Save EVERY figure the notebook shows as a PNG file too (into <results folder>/figures/), so all of them can be downloaded
import itertools, matplotlib.pyplot as plt
if not hasattr(plt, '_show_original'):
    plt._show_original = plt.show; _FIG_NO = itertools.count(1)
    def _show_and_save(*a, **k):
        os.makedirs(f'{OUT}/figures', exist_ok=True)
        for num in plt.get_fignums():
            plt.figure(num).savefig(f'{OUT}/figures/fig_{next(_FIG_NO):03d}.png', dpi=150, bbox_inches='tight')
        return plt._show_original(*a, **k)
    plt.show = _show_and_save
print('every figure will also be saved to', f'{OUT}/figures/')


## Root spot (same for all 5 methods) and the reference trajectory
* **TBI:** root region = highest hypothalamus-marker score (as in the stLearn paper); reference states hypothalamus, thalamus,
  hippocampus, injury are given to spots that clearly score as one of them (z > 1).
* **BC:** root region = the region with the largest share of DCIS/LCIS spots; root spot chosen among its DCIS spots;
  reference states DCIS/LCIS → IDC from the pathology annotation.

The reference is used **only to validate** results (section 10), never to build them.

In [ ]:
# [CELL:root]
import re
# ---- Root spot (same for all 5 methods) + the REFERENCE trajectory stated by stLearn (used only for validation) ----
#   TBI (stLearn Fig. 2): hypothalamus -> thalamus -> hippocampus -> injury penumbra; root in the hypothalamus
#   BC  (stLearn PSTS tutorial): DCIS -> invasive ductal carcinoma (IDC); root in DCIS
#   ZHUANG / HER2ST (CASCAT benchmarks): curated labels + the trajectory stated in the CASCAT paper (a TREE: states can share a time)
REF_TIME, REF_EDGES = None, None             # set per dataset; default = a chain in the order of REF_STATES
REF_SOURCE = 'stLearn paper' if DATASET in ('TBI', 'BC') else 'CASCAT paper (curated annotation)'
if DATASET in ANNOTATED:
    import networkx as _nx
    labs = ad.obs[LABEL_KEY].astype(str).values
    if DATASET == 'ZHUANG':
        code = pd.Series(labs).str.extract(r'(\d{4})')[0].values             # supertype number, e.g. '0169'
        REF_EDGES = [('0169', '0167'), ('0167', '0153'), ('0153', '0151'), ('0151', '0150'), ('0151', '0152')]
        src_ = 'CASCAT paper text'
        mn = None
        if 'milestone_network' in raw.uns: mn = pd.DataFrame(raw.uns['milestone_network'])
        elif MILESTONE_CSV: mn = pd.read_csv(MILESTONE_CSV)
        if mn is not None and {'from', 'to'} <= set(mn.columns):
            e_ = [(re.search(r'\d{4}', str(a_)), re.search(r'\d{4}', str(b_))) for a_, b_ in zip(mn['from'], mn['to'])]
            e_ = [(a_.group(), b_.group()) for a_, b_ in e_ if a_ and b_]
            if e_ and any('0169' in x for x in e_): REF_EDGES, src_ = e_, 'dataset milestone_network'
        ROOT_STATE = '0169'
        state_of = lambda j: code[j] if isinstance(code[j], str) else None
    else:                                                                    # HER2ST pathologist regions
        REF_EDGES = [('breast glands', 'immune infiltrate'), ('immune infiltrate', 'connective tissue'),
                     ('breast glands', 'tumour (in situ)'), ('tumour (in situ)', 'invasive cancer')]
        src_ = 'CASCAT paper text'
        def _her(l):
            l = l.lower()
            if 'invasive' in l: return 'invasive cancer'
            if 'gland' in l: return 'breast glands'
            if 'immune' in l: return 'immune infiltrate'
            if 'connective' in l or 'fibrous' in l: return 'connective tissue'
            if 'situ' in l or 'tumor' in l or 'tumour' in l or 'cancer' in l: return 'tumour (in situ)'
            return None                                                      # adipose / fat / undetermined: not on the trajectory
        hs = [_her(l) for l in labs]; ROOT_STATE = 'breast glands'
        state_of = lambda j: hs[j]
    G_ = _nx.Graph(REF_EDGES)
    REF_TIME = {k: float(v) for k, v in _nx.single_source_shortest_path_length(G_, ROOT_STATE).items()}
    REF_STATES = sorted(REF_TIME, key=lambda k: (REF_TIME[k], k))
    ref_state = np.array([state_of(j) if state_of(j) in REF_TIME else None for j in range(ad.n_obs)], dtype=object)
    ref_label = labs                                                         # all curated labels, for ARI / AMI (as in CASCAT)
    tmax = max(REF_TIME.values())
    ad.obs['score_injury'] = [REF_TIME[s_] / tmax if s_ is not None else np.nan for s_ in ref_state]   # target: reference time 0 -> 1
    is_root = ref_state == ROOT_STATE
    frac_root = pd.Series(is_root, index=ad.obs_names).groupby(ad.obs['region'], observed=True).agg(['mean', 'sum'])
    cl_scores = frac_root.rename(columns={'mean': f'share {ROOT_STATE}', 'sum': f'n {ROOT_STATE}'})
    cl_scores['score_injury'] = ad.obs.groupby('region', observed=True)['score_injury'].mean()   # mean reference time per region
    print(f'reference trajectory ({src_}):', REF_EDGES); print('reference time per state:', REF_TIME)
    print(f'Per region: share of root-state ({ROOT_STATE}) cells:'); display(cl_scores.round(3))
    ROOT_CLUSTER = frac_root.loc[frac_root['sum'] >= 5, 'mean'].idxmax() if (frac_root['sum'] >= 5).any() else frac_root['mean'].idxmax()
    root_pool = (ad.obs['region'] == ROOT_CLUSTER).values & is_root
    if root_pool.sum() < 5: root_pool = is_root
    TARGET_NAME = 'reference time (curated annotation)'
elif DATASET == 'TBI':
    MARKERS = {
        'hypothalamus': ['Avp', 'Oxt', 'Pmch', 'Hcrt', 'Agrp', 'Pomc', 'Gal', 'Trh', 'Ghrh', 'Cartpt'],
        'thalamus':     ['Tcf7l2', 'Prkcd', 'Slc17a6', 'Plekhg1', 'Rora', 'Zic1'],
        'hippocampus':  ['Prox1', 'C1ql2', 'Wfs1', 'Neurod6', 'Spink8', 'Fibcd1'],
        'injury':       ['Spp1', 'Lyz2', 'Cd68', 'Lgals3', 'Cd14', 'Gfap', 'Vim', 'Lcn2', 'C4b', 'Aif1', 'Hmox1'],
    }
    for name, genes in MARKERS.items():
        g = [x for x in genes if x in ad.var_names]
        if g: sc.tl.score_genes(ad, g, score_name=f'score_{name}', random_state=0)
        else: ad.obs[f'score_{name}'] = 0.0
        print(f'{name:13s} markers found: {g}')
    cl_scores = ad.obs.groupby('region', observed=True)[[f'score_{k}' for k in MARKERS]].mean()
    print('\nMean marker score per region (score = markers minus matched background genes; 0 = background):'); display(cl_scores.round(3))
    REF_STATES = ['hypothalamus', 'thalamus', 'hippocampus', 'injury']
    Zs = np.column_stack([((ad.obs[f'score_{k}'] - ad.obs[f'score_{k}'].mean()) / ad.obs[f'score_{k}'].std()).values for k in REF_STATES])
    clear = Zs.max(1) > 1.0                                   # a spot gets a reference state only if it is clearly that area (z > 1)
    # NEW root rule: stLearn starts in the hypothalamus -> use spots that CLEARLY look like hypothalamus (z > 1 and highest),
    # and the region with the largest SHARE of such spots (the old rule, highest mean score, picked a thalamus-like region)
    hyp = clear & (Zs.argmax(1) == 0)
    frac_hyp = pd.Series(hyp, index=ad.obs_names).groupby(ad.obs['region'], observed=True).mean()
    print('\nShare of clear hypothalamus spots per region:', frac_hyp.round(3).to_dict())
    ROOT_CLUSTER = frac_hyp.idxmax()
    root_pool = (ad.obs['region'] == ROOT_CLUSTER).values & hyp
    if root_pool.sum() < 5: root_pool = hyp                   # safety: too few such spots in that region -> all clear hypothalamus spots
    ref_state = np.where(clear, np.array(REF_STATES, dtype=object)[Zs.argmax(1)], None)
    ref_label = np.where(clear, np.array(REF_STATES, dtype=object)[Zs.argmax(1)], 'other')
    TARGET_NAME = 'injury score'
else:
    ann = pd.read_csv(ANNOT, sep='\t', index_col=0).reindex(ad.obs_names)
    ad.obs['annot'] = ann['annot_type'].fillna('unannotated').astype(str).values
    REF_STATES = ['DCIS/LCIS', 'IDC']
    ref_state = np.array([{'Tumor': 'DCIS/LCIS', 'Invasive': 'IDC'}.get(a) for a in ad.obs['annot']], dtype=object)
    ref_label = ad.obs['annot'].values
    ad.obs['score_injury'] = [{'DCIS/LCIS': 0.0, 'IDC': 1.0}.get(s_, np.nan) for s_ in ref_state]   # target: DCIS (0) -> IDC (1)
    cl_scores = pd.DataFrame({'score_injury': ad.obs.groupby('region', observed=True)['score_injury'].mean(),
                              'frac_DCIS': pd.Series(ref_state == 'DCIS/LCIS', index=ad.obs_names).groupby(ad.obs['region'], observed=True).mean(),
                              'frac_IDC': pd.Series(ref_state == 'IDC', index=ad.obs_names).groupby(ad.obs['region'], observed=True).mean()})
    print('Pathology annotation:', ad.obs['annot'].value_counts().to_dict())
    print('\nPer region: share of DCIS/LCIS and IDC spots:'); display(cl_scores.round(3))
    ROOT_CLUSTER = cl_scores['frac_DCIS'].idxmax()
    root_pool = ((ad.obs['region'] == ROOT_CLUSTER).values) & (ref_state == 'DCIS/LCIS')
    TARGET_NAME = 'DCIS→IDC (annotation)'
if REF_TIME is None: REF_TIME = {k: float(j) for j, k in enumerate(REF_STATES)}
if REF_EDGES is None: REF_EDGES = list(zip(REF_STATES[:-1], REF_STATES[1:]))
ad.obs['ref_state'] = pd.Categorical(ref_state, categories=REF_STATES)
ad.obs['ref_time'] = ad.obs['ref_state'].astype(object).map(REF_TIME).astype(float)
ad.obs['ref_label'] = np.asarray(ref_label, dtype=str)
print(f'reference ({REF_SOURCE}):', ', '.join(f'{a}→{b}' for a, b in REF_EDGES),
      '| cells per reference state:', ad.obs['ref_state'].value_counts().to_dict())
print('root region:', ROOT_CLUSTER)

# ---- root spot inside the root pool: stLearn's set_root rule (CytoTRACE-like: the most 'immature-looking' spot) ----
sub = ad[root_pool]
Xs = sub.X.toarray() if hasattr(sub.X, 'toarray') else np.asarray(sub.X)
n_expr = (Xs > 0).sum(1).astype(float)
Xc = Xs - Xs.mean(0); nc = n_expr - n_expr.mean()
corr = (Xc * nc[:, None]).sum(0) / (np.sqrt((Xc ** 2).sum(0) * (nc ** 2).sum()) + 1e-12)
top = np.argsort(-np.nan_to_num(corr))[:1000]
cyto = Xs[:, top].mean(1)
root = int(np.where(ad.obs_names == sub.obs_names[int(np.argmax(cyto))])[0][0])
ROOT_BARCODE = ad.obs_names[root]
ad.uns['iroot'] = root
print('root spot:', ROOT_BARCODE)

# SPACECONS 2.0: spatially diverse candidate roots inside the biologically defined root pool.
# This runs for EACH dataset because the root cell is re-executed for TBI and BC.
from sklearn.cluster import KMeans
_root_ids = np.where(np.asarray(root_pool, bool))[0]
if len(_root_ids) == 0:
    raise RuntimeError('The biological root candidate pool is empty; inspect root markers/annotations.')
_nroot = min(8, len(_root_ids))
_xyroot = np.asarray(xy[_root_ids], float)
if len(_root_ids) <= _nroot:
    _selected_roots = _root_ids.copy()
else:
    _kmroot = KMeans(n_clusters=_nroot, random_state=42, n_init=10).fit(_xyroot)
    _selected_roots=[]
    for _k in range(_nroot):
        _members=np.where(_kmroot.labels_ == _k)[0]
        if len(_members):
            _center=_kmroot.cluster_centers_[_k]
            _local=_members[np.argmin(((_xyroot[_members]-_center)**2).sum(1))]
            _selected_roots.append(_root_ids[_local])
    _selected_roots=np.asarray(_selected_roots,int)
ROOT_CANDIDATES = pd.DataFrame({
    'candidate_id':[f'R{i+1:02d}' for i in range(len(_selected_roots))],
    'spot_index':_selected_roots,
    'root_barcode':[str(ad.obs_names[i]) for i in _selected_roots],
    'x':xy[_selected_roots,0], 'y':xy[_selected_roots,1],
    'root_region':ad.obs['region'].astype(str).values[_selected_roots],
    'reference_state':[str(ad.obs['ref_state'].astype(object).values[i]) for i in _selected_roots],
})
ROOT_CANDIDATES.to_csv(f'{OUT}/multi_root_candidates.csv',index=False)
print(f'Exported {len(ROOT_CANDIDATES)} spatially distributed candidate roots to {OUT}/multi_root_candidates.csv')


fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
s = axes[0].scatter(xy[:, 0], -xy[:, 1], c=ad.obs['score_injury'], s=6, cmap='magma'); plt.colorbar(s, ax=axes[0]); axes[0].set_title(TARGET_NAME)
for k in REF_STATES:
    m = (ad.obs['ref_state'] == k).values; axes[1].scatter(xy[m, 0], -xy[m, 1], s=6, label=k)
axes[1].legend(fontsize=8, markerscale=2); axes[1].set_title(f'reference states ({REF_SOURCE})')
for lab_ in ad.obs['region'].cat.categories:
    m = (ad.obs['region'] == lab_).values; axes[2].scatter(xy[m, 0], -xy[m, 1], s=6, label=lab_)
axes[2].legend(fontsize=7, markerscale=2, ncol=2); axes[2].set_title('expression regions (Leiden)')
for a in axes:
    a.scatter(xy[root, 0], -xy[root, 1], c='cyan', s=200, marker='*', edgecolors='k'); a.set_aspect('equal'); a.axis('off')
plt.tight_layout(); plt.show()

def save_pt(col, name):
    pd.DataFrame({'barcode': ad.obs_names, 'x': xy[:, 0], 'y': xy[:, 1], 'region': ad.obs['region'].values,
                  'pseudotime': ad.obs[col].values, 'method': name}).to_csv(f'{OUT}/pt_{name}.csv', index=False)
    print(name, '| root region mean:', round(ad.obs.loc[ad.obs['region'] == ROOT_CLUSTER, col].mean(), 3),
          f'| corr. with {TARGET_NAME}:', round(spearmanr(ad.obs[col], ad.obs['score_injury'])[0], 3))

MS_GRAPH = {}       # each method's OWN milestone graph: list of (milestone a, milestone b, weight)
def paga_edges(a, key):
    C_ = a.uns['paga']['connectivities'].toarray(); cats = list(a.obs[key].cat.categories)
    return [(str(cats[i]), str(cats[j]), float(C_[i, j])) for i in range(len(cats)) for j in range(i + 1, len(cats)) if C_[i, j] > 0]


## 1. SpaceFlow (graph neural network + spatial regularisation, then DPT from the root)

In [ ]:
# [CELL:spaceflow]
from SpaceFlow import SpaceFlow
import networkx as nx, scipy.sparse as sp
# compatibility fix: SpaceFlow 1.0.4 calls a networkx function that newer networkx removed
if not hasattr(nx, 'to_scipy_sparse_matrix'):
    nx.to_scipy_sparse_matrix = lambda G, format='csr', **kw: sp.csr_matrix(nx.to_scipy_sparse_array(G, format=format, **kw))

EPOCHS = 1000          # SpaceFlow default; early stopping usually ends sooner
sf = SpaceFlow.SpaceFlow(adata=raw.copy())
sf.preprocessing_data(n_top_genes=min(2000, raw.n_vars))          # normalise, log, HVGs, PCA, spatial graph (alpha complex)
emb = sf.train(embedding_save_filepath=f'{OUT}/spaceflow_embedding.tsv',
               spatial_regularization_strength=0.1, z_dim=50, lr=1e-3,
               epochs=EPOCHS, max_patience=50, min_stop=100, random_seed=42, gpu=0)
print('embedding shape:', emb.shape)

e = sc.AnnData(emb); e.obs_names = ad.obs_names
sc.pp.neighbors(e, n_neighbors=20, use_rep='X')
sc.tl.diffmap(e, n_comps=15)
e.uns['iroot'] = int(root)
sc.tl.dpt(e)
ad.obs['t_spaceflow'] = rank01(e.obs['dpt_pseudotime'].values)
save_pt('t_spaceflow', 'spaceflow')
# ---- SpaceFlow's own milestones: its spatial domains (SpaceFlow segmentation = Leiden on the embedding) + PAGA between them ----
e2 = e.copy(); sc.pp.neighbors(e2, n_neighbors=50, use_rep='X')
sc.tl.leiden(e2, resolution=1.0, key_added='m', flavor='igraph', n_iterations=2, directed=False, random_state=0)
sc.tl.paga(e2, groups='m')
ad.obs['m_spaceflow'] = e2.obs['m'].astype(str).values; MS_GRAPH['SpaceFlow'] = paga_edges(e2, 'm')
print('SpaceFlow milestones:', ad.obs['m_spaceflow'].nunique())

# ---- SPACECONS 2.1: real multi-root rerun for SpaceFlow. Only the DPT step depends on the root, so it is cheap. ----
if 'ROOT_CANDIDATES' in globals() and len(ROOT_CANDIDATES):
    _e_mr, _rows = e.copy(), []
    for _rb, _ri in zip(ROOT_CANDIDATES['root_barcode'], ROOT_CANDIDATES['spot_index']):
        _e_mr.uns['iroot'] = int(_ri); sc.tl.dpt(_e_mr)
        _rows.append(pd.DataFrame({'root_barcode': _rb, 'method': 'SpaceFlow', 'barcode': ad.obs_names.astype(str),
                                   'pseudotime': _e_mr.obs['dpt_pseudotime'].values}))
    pd.concat(_rows).to_csv(f'{OUT}/multi_root_spaceflow_pseudotime.csv', index=False)
    print(f'SpaceFlow re-run from {len(_rows)} candidate roots -> multi_root_spaceflow_pseudotime.csv')
    del _e_mr, _rows


## 2. SpaTrack (optimal transport: cost = expression distance + physical distance)

In [ ]:
# [CELL:spatrack]
import numpy as np
if not hasattr(np, 'Inf'):
    np.Inf = np.inf          # compatibility fix: spaTrack 0.1.1 uses a name removed in NumPy 2
import spaTrack as spt

ALPHA_SPACE, ALPHA_EXPR = 1, 1
st = ad.copy()
st.obsm['X_spatial'] = xy
st.obs['cluster'] = ad.obs['region'].astype(str)
st.obsp['trans'] = spt.get_ot_matrix(st, data_type='spatial', alpha1=ALPHA_SPACE, alpha2=ALPHA_EXPR)
start_cells = spt.set_start_cells(st, select_way='coordinates', start_point=tuple(xy[root]),
                                  basis='spatial', n_neigh=20)
print('start cells:', len(start_cells), '| their tissues:', st.obs['region'].iloc[list(start_cells)].value_counts().to_dict())
ad.obs['t_spatrack'] = rank01(spt.get_ptime(st, list(start_cells)))
save_pt('t_spatrack', 'spatrack')
# ---- SpaTrack's own milestones: communities of its transport (transition) graph; edges = transported mass between them ----
import scipy.sparse as sp
# The OT transition matrix is DENSE (every spot -> every spot), so Leiden on it is extremely slow on TBI.
# Keep only each spot's K_TR strongest transitions (a sparse 'transport kNN graph'); flows between milestones still use the full matrix.
K_TR = 15
Td = st.obsp['trans']; Td = Td.toarray() if sp.issparse(Td) else np.asarray(Td, dtype=float)
top_ = np.argpartition(-Td, K_TR, axis=1)[:, :K_TR]; rr_ = np.repeat(np.arange(Td.shape[0]), K_TR)
Tk = sp.csr_matrix((Td[rr_, top_.ravel()], (rr_, top_.ravel())), shape=Td.shape)
Asym = ((Tk + Tk.T) / 2).tocsr(); Asym.eliminate_zeros()
sc.tl.leiden(st, adjacency=Asym, resolution=1.0, key_added='m', flavor='igraph', n_iterations=2, directed=False, random_state=0)
labm = st.obs['m'].astype(str).values; ms_ = sorted(set(labm), key=int)
H = sp.csr_matrix((np.ones(len(labm)), (np.arange(len(labm)), [ms_.index(x) for x in labm])), shape=(len(labm), len(ms_)))
F = np.asarray(H.T @ (Td @ H)); F = F + F.T; np.fill_diagonal(F, 0); F = F / (F.max() + 1e-12)
ad.obs['m_spatrack'] = labm
# ---- SPACECONS-3 exports: SpaTrack's transport plan on spatially close pairs (a similarity view) + a per-spot confidence ----
from sklearn.neighbors import NearestNeighbors as _NN
_nb15 = _NN(n_neighbors=16).fit(xy).kneighbors(xy)[1]; _r15 = np.repeat(np.arange(len(xy)), 15); _c15 = _nb15[:, 1:].ravel()
sp.save_npz(f'{OUT}/spatrack_ot_spatial15.npz', sp.csr_matrix((Td[_r15, _c15], (_r15, _c15)), shape=Td.shape))
_Pr = Td / (Td.sum(1, keepdims=True) + 1e-300); _H = -np.where(_Pr > 0, _Pr * np.log(_Pr + 1e-300), 0).sum(1)
np.save(f'{OUT}/spatrack_confidence.npy', 0.5 + 0.5 * rank01(-_H))     # focused transport (low entropy) = more confident
del _Pr, _H
MS_GRAPH['SpaTrack'] = [(ms_[i], ms_[j], float(F[i, j])) for i in range(len(ms_)) for j in range(i + 1, len(ms_)) if F[i, j] > 1e-3]
print('SpaTrack milestones:', len(ms_))


## 3. StaVia (spatial smoothing + spatial kNN, random walks with memory)

In [ ]:
# [CELL:stavia]
import json
# ---- compatibility fixes (pyVIA was written for NumPy 1 and older SciPy) ----
import numpy as np, scipy.sparse as sp
for old, new in {'bool8':'bool_','object0':'object_','int0':'intp','uint0':'uintp','float_':'float64',
                 'complex_':'complex128','unicode_':'str_','string_':'bytes_','longfloat':'longdouble',
                 'singlecomplex':'complex64','cfloat':'complex128','longcomplex':'clongdouble',
                 'clongfloat':'clongdouble','void0':'void','bytes0':'bytes_','str0':'str_'}.items():
    if old not in np.__dict__: setattr(np, old, getattr(np, new))
for old, val in {'Inf':np.inf,'NaN':np.nan,'Infinity':np.inf,'NINF':-np.inf,'PINF':np.inf}.items():
    if old not in np.__dict__: setattr(np, old, val)
import matplotlib, matplotlib.cm as cm
if not hasattr(cm, 'get_cmap'): cm.get_cmap = matplotlib.colormaps.get_cmap   # removed in new matplotlib
import pyVIA.utils_via as uv, pyVIA.core as vc
def _sparse_from_igraph(graph, weight_attr=None):            # fixed copy of pyVIA's helper (new SciPy rejects zip())
    edges = graph.get_edgelist()
    w = list(graph.es[weight_attr]) if weight_attr else [1] * len(edges)
    if not graph.is_directed():
        edges = edges + [(b, a) for a, b in edges]; w = w + w
    n = graph.vcount()
    if not edges: return sp.csr_matrix((n, n))
    r, c = map(np.asarray, zip(*edges))
    return sp.csr_matrix((np.asarray(w, float), (r, c)), shape=(n, n))
uv.get_sparse_from_igraph = vc.get_sparse_from_igraph = _sparse_from_igraph
from pyVIA.core import VIA
from pyVIA.utils_via import spatial_input
from sklearn.decomposition import PCA

# ---- Method 5: StaVia ----
SPATIAL_WEIGHT = 0.3     # how much each spot's expression is mixed with its 8 spatial neighbours (paper uses 0.3 to 0.5)
Xh = ad[:, ad.var['highly_variable']].X                      # log-normalised HVG expression
Xh = Xh.toarray() if sp.issparse(Xh) else np.asarray(Xh)
Xs = spatial_input(Xh, xy.astype(float), knn_spatial=8, spatial_weight=SPATIAL_WEIGHT)   # step 1: spatial smoothing
Xs = Xs.toarray() if sp.issparse(Xs) else np.asarray(Xs)
pcs_sv = PCA(30, random_state=0).fit_transform((Xs - Xs.mean(0)) / (Xs.std(0) + 1e-8))

v = VIA(data=pcs_sv, true_label=list(ad.obs['region'].astype(str)), root_user=[int(root)],
        knn=30, memory=5, random_seed=42, num_threads=4,
        spatial_coords=xy.astype(float), do_spatial_knn=True, spatial_knn=15)                  # step 2: spatial kNN
v.run_VIA()

pt = np.asarray(v.single_cell_pt_markov, dtype=float)
assert len(pt) == ad.n_obs, f'StaVia returned {len(pt)} values for {ad.n_obs} spots'
ad.obs['t_stavia'] = rank01(pt)
ad.obs['stavia_cluster'] = pd.Categorical(np.asarray(v.labels).astype(str))
print('StaVia clusters:', ad.obs['stavia_cluster'].nunique(), '| terminal clusters:', v.terminal_clusters)
save_pt('t_stavia', 'stavia')
ad.obs['m_stavia'] = ad.obs['stavia_cluster'].astype(str).values          # StaVia's own milestones
MS_GRAPH['StaVia'] = [(str(a_), str(b_), 1.0) for a_, b_ in v.edgelist_maxout]  # StaVia's own cluster graph
json.dump({'terminal_clusters': [str(c_) for c_ in (v.terminal_clusters or [])]}, open(f'{OUT}/stavia_terminal.json', 'w'))   # SPACECONS-3: end points

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
s = axes[0].scatter(xy[:, 0], xy[:, 1], c=ad.obs['t_stavia'], s=5, cmap='viridis'); plt.colorbar(s, ax=axes[0])
axes[0].scatter(xy[root, 0], xy[root, 1], c='red', s=100, marker='*'); axes[0].set_title('StaVia pseudotime')
for k in ad.obs['stavia_cluster'].cat.categories:
    m = (ad.obs['stavia_cluster'] == k).values; axes[1].scatter(xy[m, 0], xy[m, 1], s=5, label=k)
axes[1].legend(fontsize=7, markerscale=3, ncol=2); axes[1].set_title('StaVia clusters')
for a in axes: a.set_aspect('equal')
plt.tight_layout(); plt.show()


## 4. stLearn PSTS (full version, with the H&E image)
Steps, as in the stLearn tutorial: normalise/log → PCA → **cut an image tile around each spot and describe it with a ResNet50 CNN** →
**stSME** (smooth each spot with neighbours that are close in space, look alike in the image and in expression) → PCA → Louvain clusters →
**PSTS**: diffusion pseudotime from our root on the stSME graph + spatial sub-clusters, and the cluster-level trajectory.
The next cell saves the script; the cell after runs it in the Python 3.11 environment.

In [ ]:
%%writefile run_stlearn_psts.py
import os; os.environ['MPLBACKEND'] = 'Agg'   # this runs outside Jupyter: never inherit the notebook's inline plotting backend
"""stLearn PSTS (Pham et al., Nat Commun 2023) following the official stLearn tutorial.
Runs in its own Python 3.11 environment (stLearn needs Python < 3.13 and NumPy < 2).
Input : EITHER a 10x Visium folder (filtered_feature_bc_matrix.h5 + spatial/ with the H&E image)  -> full stSME (image + space + expression)
        OR an .h5ad with counts in X and coordinates in obsm['spatial'] (MERFISH / ST without a registered image)
           -> stLearn without the morphology step (stSME needs image tiles), everything else identical.
Output: CSV with one stLearn pseudotime value per spot (+ its louvain cluster) and a JSON with the run information."""

import argparse, os, json, numpy as np, pandas as pd
ap = argparse.ArgumentParser()
ap.add_argument('--visium_dir', default=None); ap.add_argument('--h5ad', default=None)
ap.add_argument('--root_barcode', required=True)
ap.add_argument('--keep', required=True, help='text file with barcodes to keep (same spots as the other methods)')
ap.add_argument('--out', required=True); ap.add_argument('--tile_dir', default='/tmp/stl_tiles')
ap.add_argument('--resolution', type=float, default=1.0); ap.add_argument('--eps', type=float, default=None,
                help='DBSCAN radius for spatial sub-clusters; default 50 (Visium image pixels) or 3 x median neighbour distance (h5ad)')
ap.add_argument('--no_imagenet_weights', action='store_true', help='testing only: offline, use untrained CNN')
a = ap.parse_args()
assert (a.visium_dir is None) != (a.h5ad is None), 'give exactly one of --visium_dir or --h5ad'

if a.no_imagenet_weights:                      # offline test only; on Kaggle the ImageNet ResNet50 weights download normally
    import keras.applications.resnet50 as kr
    _R = kr.ResNet50; kr.ResNet50 = lambda *x, **k: _R(*x, **{**k, 'weights': None})
import stlearn as st, scanpy as sc

keep = [l.strip() for l in open(a.keep) if l.strip()]
if a.visium_dir:
    data = st.Read10X(a.visium_dir); mode = 'visium + H&E (stSME)'
else:
    data = sc.read_h5ad(a.h5ad); mode = 'h5ad, no image (stLearn without the morphology step)'
    xy = np.asarray(data.obsm['spatial'], float)[:, :2]
    data.obs['imagecol'], data.obs['imagerow'] = xy[:, 0], xy[:, 1]          # stLearn reads spot positions from these columns
data.var_names_make_unique()
data = data[data.obs_names.isin(keep)].copy()
data.layers['raw_count'] = data.X.copy()

# 1) preprocessing (stLearn tutorial)
st.pp.filter_genes(data, min_cells=3)
st.pp.normalize_total(data); st.pp.log1p(data)
st.em.run_pca(data, n_comps=min(50, data.n_vars - 1), random_state=0)

if a.visium_dir:
    # 2) morphology: cut an image tile around every spot and describe it with a ResNet50 CNN
    st.pp.tiling(data, a.tile_dir)
    st.pp.extract_feature(data)
    # 3) stSME: smooth each spot with neighbours that are close in space AND look alike in the image AND in expression
    st.spatial.SME.SME_normalize(data, use_data='raw', weights='weights_matrix_all')
    d = data.copy(); d.X = d.obsm['raw_SME_normalized']
    st.pp.scale(d); st.em.run_pca(d, n_comps=50, random_state=0)
    eps = a.eps or 50
else:
    d = data                                    # no image -> no stSME; PCA of the normalised expression is used directly
    from sklearn.neighbors import NearestNeighbors
    dist, _ = NearestNeighbors(n_neighbors=2).fit(xy).kneighbors(xy)
    eps = a.eps or float(3 * np.median(dist[:, 1]))
st.pp.neighbors(d, n_neighbors=25, use_rep='X_pca', random_state=0)
st.tl.clustering.louvain(d, resolution=a.resolution, random_state=0)

# 4) PSTS: root = the same root spot used by all methods; diffusion pseudotime + spatial sub-clusters
root_idx = int(np.where(d.obs_names == a.root_barcode)[0][0])
d.uns['iroot'] = root_idx
st.spatial.trajectory.pseudotime(d, eps=eps, use_rep='X_pca', use_label='louvain', max_nodes=5)   # allow 4-cluster paths like the paper's
root_cl = str(d.obs['louvain'].iloc[root_idx])
info = {'mode': mode, 'eps': eps, 'n_spots': int(d.n_obs), 'n_louvain': int(d.obs['louvain'].nunique()), 'root_cluster': root_cl}

# 5) PSTS global trajectory (cluster level), as in the paper: a path of connected clusters starting at the root's cluster.
paths = [p for p in d.uns.get('available_paths', {}).values() if str(p[0]) == root_cl and len(p) > 1]
cl_pt = d.obs.groupby('louvain', observed=True)['dpt_pseudotime'].mean()
paths.sort(key=lambda p: (len(p), cl_pt.get(str(p[-1]), 0)), reverse=True)
for p in paths:
    try:
        st.spatial.trajectory.pseudotimespace_global(d, use_label='louvain', list_clusters=[str(c) for c in p])
        info['psts_path'] = [str(c) for c in p]; info.pop('psts_global_error', None); break
    except Exception as e:
        info['psts_global_error'] = repr(e)[:200]
else:
    info.setdefault('psts_global_error', 'no available path from the root cluster')

if 'paga' not in d.uns or d.uns['paga'].get('groups') != 'louvain': sc.tl.paga(d, groups='louvain')
Cg = d.uns['paga']['connectivities'].toarray(); cats = [str(c) for c in d.obs['louvain'].cat.categories]
info['paga_edges'] = [[cats[i], cats[j], float(Cg[i, j])] for i in range(len(cats)) for j in range(i + 1, len(cats)) if Cg[i, j] > 0]
pd.DataFrame({'barcode': d.obs_names, 'stlearn_pseudotime': d.obs['dpt_pseudotime'].values,
              'stlearn_cluster': d.obs['louvain'].astype(str).values}).to_csv(a.out, index=False)
json.dump(info, open(a.out.replace('.csv', '_info.json'), 'w'), indent=1)
np.save(a.out.replace('.csv', '_pca.npy'), np.asarray(d.obsm['X_pca'], dtype=np.float32))   # SPACECONS-3: stLearn's (stSME) view
print('stLearn PSTS done:', {k: v for k, v in info.items() if k != 'paga_edges'})


In [ ]:
# [CELL:stlearn-run]
# helper: run a separate program and SHOW its progress live (instead of only at the end), with elapsed time
import subprocess, time, os, sys
def run_live(cmd, skip=lambda l: False, every=1):
    t0 = time.time(); env = {**os.environ, 'MPLBACKEND': 'Agg', 'PYTHONUNBUFFERED': '1'}
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, bufsize=1)
    tail, k = [], 0
    for line in p.stdout:
        line = line.rstrip(); tail = (tail + [line])[-40:]
        if not line or skip(line): continue
        k += 1
        if k % every == 0: print(f'[{(time.time() - t0) / 60:5.1f} min] {line[:200]}', flush=True)
    p.wait(); print(f'--- finished in {(time.time() - t0) / 60:.1f} min, exit code {p.returncode}')
    return p.returncode, '\n'.join(tail)

import json
# ---- run stLearn PSTS in its own environment, with live progress (skipped if a result for these spots already exists) ----
if not os.path.exists(f'{OUT}/raw_stlearn.csv'):
    if VISIUM_DIR:                                   # Visium + H&E: full stSME
        src_args = ['--visium_dir', VISIUM_DIR]
    else:                                            # MERFISH / HER2ST h5ad: no registered image -> stLearn without morphology
        sin = raw.copy(); sin.obsm = {'spatial': xy.copy()}; sin.uns = {}
        sin.obs = sin.obs[[]]; sin.write(f'{OUT}/stlearn_input.h5ad'); src_args = ['--h5ad', f'{OUT}/stlearn_input.h5ad']
    cmd = [STL_PY, 'run_stlearn_psts.py', *src_args, '--root_barcode', ROOT_BARCODE,
           '--keep', f'{OUT}/kept_spots.txt', '--out', f'{OUT}/raw_stlearn.csv', '--tile_dir', '/tmp/stl_tiles']
    rc, tail = run_live(cmd, skip=lambda l: ('it/s' in l or 's/it' in l) and '100%' not in l)   # hide progress-bar spam
    if rc != 0: print(tail); print('stLearn did not finish -> skipped (the comparison will run without it)')
if os.path.exists(f'{OUT}/raw_stlearn.csv'):
    res = pd.read_csv(f'{OUT}/raw_stlearn.csv', index_col='barcode', dtype={'barcode': str})   # v3.1h: cell IDs that look like numbers stay text
    print('spots returned:', len(res), 'of', ad.n_obs)
    ad.obs['t_stlearn'] = rank01(res['stlearn_pseudotime'].reindex(ad.obs_names).values)
    ad.obs['stlearn_cluster'] = res['stlearn_cluster'].reindex(ad.obs_names).astype(str).values
    info = json.load(open(f'{OUT}/raw_stlearn_info.json')); print({k: v for k, v in info.items() if k != 'paga_edges'})
    ad.obs['m_stlearn'] = ad.obs['stlearn_cluster']                          # stLearn's own milestones (stSME clusters)
    MS_GRAPH['stLearn'] = [tuple(e_) for e_ in info.get('paga_edges', [])]   # stLearn's PAGA graph between them
    save_pt('t_stlearn', 'stlearn')


## 5. CASCAT (causal trajectories; Yu et al., NAR 2025)
Run exactly as in the authors' README (`main.py --mode train`, which trains and then infers), with the authors' **Visium settings** (`HER2ST.yml`).
CASCAT needs region labels (`obs['cluster']`) — we give it our Leiden regions — and the root: our root cluster and root spot.
Its per-spot output is `cascat_pseudotime`.

**Runtime note:** CASCAT's training is quick (~1 min per trial); almost all of its time is the **causal-tree step**, which computes a conditional-mutual-information value for every pair of regions at every branch.
With the authors' kernel-density estimator this takes **hours** on Kaggle for 16 regions. By default this notebook uses CASCAT's own **binned CMI estimator** (`CASCAT_KDE = 0`, a few minutes), 2 trials × 50 epochs, and a **60-minute time limit** after which CASCAT is skipped. Set `CASCAT_KDE = 1` and `5, 100` to reproduce the authors' exact settings.

In [ ]:
%%writefile run_cascat.py
import os; os.environ['MPLBACKEND'] = 'Agg'   # this runs outside Jupyter: never inherit the notebook's inline plotting backend
"""CASCAT (Yu et al., Nucleic Acids Research 2025) on one spatial section, following the authors' README.
Runs inside the CASCAT folder, in its own Python 3.10 environment.
Input : an h5ad with raw counts in X, obsm['spatial'], obs['cluster'] (region labels) and uns['start_id'] (root spot).
Output: CSV with CASCAT's per-spot pseudotime (adata.obs['cascat_pseudotime']) and its clusters."""
import argparse, os, sys, subprocess, yaml, glob
ap = argparse.ArgumentParser()
ap.add_argument('--cascat_dir', required=True); ap.add_argument('--h5ad', required=True)
ap.add_argument('--root_cluster', required=True); ap.add_argument('--out', required=True)
ap.add_argument('--work', default='/tmp/cascat_run'); ap.add_argument('--device', default='cuda')
ap.add_argument('--ntrials', type=int, default=5); ap.add_argument('--epochs', type=int, default=100)
ap.add_argument('--config', default='HER2ST.yml', help='authors YAML in config/yaml/CMI/');
ap.add_argument('--data_name', default='TBI')
ap.add_argument('--kde', type=int, default=1, help='1 = authors default (kernel-density CMI in the tree step, slow); 0 = CASCAT binned CMI estimator (fast)')
a = ap.parse_args()

cfg = yaml.safe_load(open(os.path.join(a.cascat_dir, 'config/yaml/CMI', a.config)))   # authors' settings for this kind of data
cfg.update({'data_name': a.data_name, 'adata_file': os.path.abspath(a.h5ad), 'root': str(a.root_cluster),
            'device': a.device, 'ntrials': a.ntrials, 'epochs': a.epochs, 'species': 'Mouse',
            'img_dir': f'{a.work}/img/', 'clu_dir': f'{a.work}/clu/', 'output_dir': f'{a.work}/result/',
            'CMI_dir': f'{a.work}/cmi/'})
cfg.setdefault('ncomp', 15)
if cfg.get('percent') is None: cfg['percent'] = 0.07   # v3.1e: Zhuang YAML has no 'percent'; newer CASCAT code needs it (same 7% as HER2ST.yml)                      # diffusion components for pseudotime (value from the authors' tree1.yml)
os.makedirs(a.work, exist_ok=True); os.makedirs(cfg['img_dir'], exist_ok=True)
yml = f'{a.work}/{a.data_name}.yml'; yaml.safe_dump(cfg, open(yml, 'w'))
print('CASCAT config:', a.config, {k: cfg.get(k) for k in ['a_k', 'percent', 'epochs', 'ntrials', 'root', 'hvg', 'norm_target', 'device']})

# CASCAT hard-codes kde=True in infer.py; make it follow --kde (authors' own alternative estimator in BinaryCMITree)
inf = os.path.join(a.cascat_dir, 'infer.py'); src = open(inf).read()
if 'CASCAT_KDE' not in src:
    src = 'import os\n' + src.replace("save_dir=self.save_path, kde=True)", "save_dir=self.save_path, kde=os.environ.get('CASCAT_KDE', '1') == '1')")
    open(inf, 'w').write(src)
os.environ['CASCAT_KDE'] = str(a.kde)
r = subprocess.run([sys.executable, 'main.py', '--YAML', yml, '--mode', 'train', '--verbose', 'True'], cwd=a.cascat_dir, env={**os.environ, 'MPLBACKEND': 'Agg'})
if r.returncode != 0: sys.exit(r.returncode)

import scanpy as sc, pandas as pd
res = sc.read_h5ad(os.path.join(cfg['output_dir'], 'data_processed.h5ad'))
pd.DataFrame({'barcode': res.obs_names, 'cascat_pseudotime': res.obs['cascat_pseudotime'].values,
              'cascat_cluster': res.obs['cascat_clusters'].astype(str).values}).to_csv(a.out, index=False)
print('CASCAT done:', res.n_obs, 'spots | root cluster:', res.uns.get('root'),
      '| tree:', res.uns['cascat_connectivities'].shape if 'cascat_connectivities' in res.uns else None)


In [ ]:
# [CELL:cascat-run]
import os, sys, time, subprocess, shutil, glob, torch
# ---- settings ----
CASCAT_TRIALS, CASCAT_EPOCHS = 2, 50   # authors' default 5 x 100 (training is quick; the slow part is the causal tree below)
CASCAT_KDE = 0                         # 0 = CASCAT's binned CMI estimator in the tree step (fast, minutes); 1 = authors' kernel-density CMI (hours)
CASCAT_MAX_MIN = 60                    # time limit in minutes: if exceeded, CASCAT is stopped and skipped so the notebook continues

def run_live(cmd, skip=lambda l: False, timeout_min=None, done_file=None):
    # prints the tool's output live; stops at the time limit, or 2 min after done_file appears with no new output (avoids the end-of-run hang)
    import threading, queue
    t0 = time.time(); env = {**os.environ, 'MPLBACKEND': 'Agg', 'PYTHONUNBUFFERED': '1'}
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, bufsize=1)
    q = queue.Queue(); threading.Thread(target=lambda: [q.put(l) for l in p.stdout], daemon=True).start()
    tail, last = [], time.time()
    while True:
        try:
            line = q.get(timeout=5).rstrip(); last = time.time(); tail = (tail + [line])[-40:]
            if line and not skip(line): print(f'[{(time.time() - t0) / 60:5.1f} min] {line[:200]}', flush=True)
            continue
        except queue.Empty: pass
        if p.poll() is not None: break
        if timeout_min and time.time() - t0 > timeout_min * 60:
            p.kill(); print(f'--- stopped: time limit of {timeout_min} min reached'); return -9, '\n'.join(tail)
        if done_file and os.path.exists(done_file) and time.time() - last > 120:
            p.kill(); subprocess.run('pkill -f "main.py --YAML"', shell=True)
            print('--- result saved but process did not exit; stopped it'); return 0, '\n'.join(tail)
    print(f'--- finished in {(time.time() - t0) / 60:.1f} min, exit code {p.returncode}')
    return p.returncode, '\n'.join(tail)

if not os.path.exists(f'{OUT}/raw_cascat.csv'):
    cin = raw.copy(); cin.obs = cin.obs[[]]
    cin.obs['cluster'] = ad.obs['region'].astype(str).values
    cin.obsm['spatial'] = xy.copy(); cin.uns = {'start_id': ROOT_BARCODE}
    cin.write(f'{OUT}/cascat_input.h5ad')
    cmd = [CAS_PY, 'run_cascat.py', '--cascat_dir', CASCAT_DIR, '--h5ad', f'{OUT}/cascat_input.h5ad',
           '--root_cluster', str(ROOT_CLUSTER), '--out', f'{OUT}/raw_cascat.csv', '--work', f'/tmp/cascat_run_{DATASET}',
           '--device', 'cuda' if torch.cuda.is_available() else 'cpu',
           '--ntrials', str(CASCAT_TRIALS), '--epochs', str(CASCAT_EPOCHS), '--kde', str(CASCAT_KDE),
           '--config', {'ZHUANG': 'Zhuang-ABCA-2.005.yml'}.get(DATASET, 'HER2ST.yml'), '--data_name', DATASET]   # authors' config per data type
    ep = [0]
    def quiet(l):
        if l.startswith('Epoch'): ep[0] += 1; return ep[0] % 10 != 1
        if l.startswith('leaf_i'): ep[0] += 1; return ep[0] % 20 != 1          # show every 20th tree calculation
        return ('Warning' in l) or ('warnings.warn' in l) or ('it/s' in l and '100%' not in l)
    CASCAT_WORK = f'/tmp/cascat_run_{DATASET}'
    subprocess.run(f'rm -rf {CASCAT_WORK}', shell=True)                       # v3.1 fix: never re-use another dataset's cached CMI.csv
    rc, tail = run_live(cmd, skip=quiet, timeout_min=CASCAT_MAX_MIN, done_file=f'/tmp/cascat_run_{DATASET}/result/data_processed.h5ad')
    if rc != 0: print(tail[-2000:]); print('CASCAT did not finish -> skipped (the comparison will run without it)')
    if os.path.exists(f'/tmp/cascat_run_{DATASET}/result/data_processed.h5ad'):
        shutil.copy(f'/tmp/cascat_run_{DATASET}/result/data_processed.h5ad', f'{OUT}/cascat_result.h5ad')   # keep CASCAT's tree
    _cmi = sorted(glob.glob(f'/tmp/cascat_run_{DATASET}/cmi/**/CMI.csv', recursive=True), key=os.path.getmtime)
    if _cmi: shutil.copy(_cmi[-1], f'{OUT}/cascat_CMI.csv')                     # SPACECONS-3: CASCAT's causal link scores
h5_res = f'/tmp/cascat_run_{DATASET}/result/data_processed.h5ad'
if not os.path.exists(f'{OUT}/raw_cascat.csv') and os.path.exists(h5_res):     # rescue: build the CSV from CASCAT's saved result
    r_ = sc.read_h5ad(h5_res)
    pd.DataFrame({'barcode': r_.obs_names, 'cascat_pseudotime': r_.obs['cascat_pseudotime'].values,
                  'cascat_cluster': r_.obs['cascat_clusters'].astype(str).values}).to_csv(f'{OUT}/raw_cascat.csv', index=False)
if os.path.exists(f'{OUT}/raw_cascat.csv'):
    res = pd.read_csv(f'{OUT}/raw_cascat.csv', index_col='barcode', dtype={'barcode': str})
    ad.obs['t_cascat'] = rank01(res['cascat_pseudotime'].reindex(ad.obs_names).values)
    ad.obs['cascat_cluster'] = res['cascat_cluster'].reindex(ad.obs_names).astype(str).values
    save_pt('t_cascat', 'cascat')
    ad.obs['m_cascat'] = ad.obs['cascat_cluster']                            # CASCAT's own milestones
    h5c = f'{OUT}/cascat_result.h5ad' if os.path.exists(f'{OUT}/cascat_result.h5ad') else f'/tmp/cascat_run_{DATASET}/result/data_processed.h5ad'
    if os.path.exists(h5c):
        Tc = sc.read_h5ad(h5c).uns['cascat_connectivities']; Tc = Tc if isinstance(Tc, pd.DataFrame) else pd.DataFrame(Tc)
        MS_GRAPH['CASCAT'] = [(str(r_), str(c_), 1.0) for r_ in Tc.index for c_ in Tc.columns
                              if str(r_) < str(c_) and (Tc.loc[r_, c_] > 0 or Tc.loc[c_, r_] > 0)]   # CASCAT's causal tree



## 6. Comparison
1. **Biological check** — does each method start in the root (hypothalamus) cluster and increase towards the injury?
2. **Mean pseudotime per region**, regions sorted from least to most injured.
3. **Agreement matrix** (Spearman) and **M_eff** (how many truly different opinions).
4. **Maps** of every method (cyan star = root).

In [ ]:
# [CELL:comparison]
METHODS = [  # column, name, family
    ('t_spaceflow', 'SpaceFlow', 'Graph neural net (DGI) + DPT'),
    ('t_spatrack',  'SpaTrack',  'Optimal transport (expression + distance)'),
    ('t_stavia',    'StaVia',    'Random walks with memory + spatial smoothing'),
    ('t_stlearn',   'stLearn',   'stSME (space + image + expression) + PSTS'),
    ('t_cascat',    'CASCAT',    'Causal (CMI) pruned spatial graph + tree'),
]
METHODS = [m for m in METHODS if m[0] in ad.obs]
cols = [m[0] for m in METHODS]; names = [m[1] for m in METHODS]
print(f'{len(cols)} methods ran:', ', '.join(names))

# 1) biological check: the trajectory should START in the root region and END in the region with the highest target value
chk = pd.DataFrame({
    'root cluster mean pt':   [ad.obs.loc[ad.obs['region'] == ROOT_CLUSTER, c].mean() for c in cols],
    'end region mean pt': [ad.obs.loc[ad.obs['region'] == cl_scores['score_injury'].idxmax(), c].mean() for c in cols],
    f'Spearman with {TARGET_NAME}': [spearmanr(ad.obs[c], ad.obs['score_injury'])[0] for c in cols],
    'family': [m[2] for m in METHODS]}, index=names).round(3)
print(f'\n1) Does each method run from the root region {ROOT_CLUSTER} (should be ≈0) towards the region with the highest {TARGET_NAME} (should be ≈1)?'); display(chk)

# 2) mean pseudotime of every region cluster, clusters sorted from least to most injured
order = cl_scores['score_injury'].sort_values().index
reg = ad.obs.groupby('region', observed=True)[cols].mean().loc[order]; reg.columns = names
reg.insert(0, TARGET_NAME, cl_scores.loc[order, 'score_injury'].values)
print(f'\n2) Mean pseudotime per region cluster (rows: lowest -> highest {TARGET_NAME}):'); display(reg.round(3))

# 3) agreement + M_eff
A = ad.obs[cols].corr(method='spearman'); A.index = A.columns = names
lam = np.clip(np.linalg.eigvalsh(A.values), 0, None); meff = lam.sum() ** 2 / (lam ** 2).sum()
fig, ax = plt.subplots(figsize=(1.1 * len(names) + 2, 1.1 * len(names) + 1))
im = ax.imshow(A.values, cmap='viridis', vmin=min(0, A.values.min()), vmax=1)
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=45, ha='right')
ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
for i in range(len(names)):
    for j in range(len(names)):
        ax.text(j, i, f'{A.values[i, j]:.2f}', ha='center', va='center', color='black' if A.values[i, j] > 0.6 else 'white')
plt.colorbar(im, ax=ax, fraction=0.046, label='Spearman agreement'); ax.set_title(f'3) Agreement   (M_eff = {meff:.2f} of {len(names)})')
plt.tight_layout(); plt.show()
red = ((A.sum(1) - 1) / (len(names) - 1)).sort_values(ascending=False).round(3)
print('M_eff =', round(meff, 2), '\nAverage agreement with the other methods:'); print(red.to_string())

# 4) maps
n = len(cols) + 2; nc = 4; nr = int(np.ceil(n / nc))
fig, axes = plt.subplots(nr, nc, figsize=(5 * nc, 4.6 * nr)); axes = np.ravel(axes)
for lab in ad.obs['region'].cat.categories:
    m = (ad.obs['region'] == lab).values; axes[0].scatter(xy[m, 0], -xy[m, 1], s=4, label=lab)
axes[0].legend(fontsize=6, markerscale=3, ncol=2); axes[0].set_title('Region clusters')
s0 = axes[1].scatter(xy[:, 0], -xy[:, 1], c=ad.obs['score_injury'], s=4, cmap='magma'); axes[1].set_title(TARGET_NAME)
for a, c, nm in zip(axes[2:], cols, names):
    s = a.scatter(xy[:, 0], -xy[:, 1], c=ad.obs[c], s=4, cmap='viridis', vmin=0, vmax=1); a.set_title(nm)
for a in axes[:n]:
    a.scatter(xy[root, 0], -xy[root, 1], c='cyan', s=150, marker='*', edgecolors='k'); a.set_aspect('equal'); a.axis('off')
for a in axes[n:]: a.axis('off')
fig.colorbar(s, ax=axes[:n].tolist(), fraction=0.02, label='pseudotime (0 = root, 1 = latest)'); plt.show()

chk.to_csv(f'{OUT}/comparison_check.csv'); reg.to_csv(f'{OUT}/comparison_regions.csv'); A.round(4).to_csv(f'{OUT}/comparison_agreement.csv')
ad.obs[['region', 'score_injury'] + cols].assign(x=xy[:, 0], y=xy[:, 1]).to_csv(f'{OUT}/all_methods_pseudotime.csv')
print('Saved:', sorted(os.path.basename(f) for f in glob.glob(f'{OUT}/*.csv')))


## 6b. Pseudotime and trajectory graph for each method
For every method, three panels (also saved as `fig_<method>.png`):
* **(a) pseudotime on the H&E image** — cyan star = root spot;
* **(b) trajectory graph on the tissue** — nodes = clusters placed at their position in the tissue, coloured by mean pseudotime, arrows from earlier to later.
  Methods that build their own graph show **their own**: StaVia's cluster graph, stLearn's PSTS path, CASCAT's causal tree.
  SpaceFlow and SpaTrack only give a per-spot ordering, so they are drawn on the shared region graph (PAGA), with arrows set by their pseudotime;
* **(c) pseudotime per region** (box plots, regions sorted from earliest to latest).

In [ ]:
# [CELL:plots]
# ---------- Per-method figures: (a) pseudotime on the H&E image, (b) trajectory graph on the tissue, (c) pseudotime per region ----------
import networkx as nx, scanpy as sc
from matplotlib.patches import FancyArrowPatch

img, pxy = tissue_image()                                                     # spot positions on the hires image

if 'paga' not in ad.uns or ad.uns['paga'].get('groups') != 'region':
    sc.tl.paga(ad, groups='region')                               # shared expression graph between regions (used if a method has no graph of its own)
PAGA_EDGES = [(ad.obs['region'].cat.categories[i], ad.obs['region'].cat.categories[j])
              for i, j in zip(*np.triu(ad.uns['paga']['connectivities'].toarray() > 0.1, 1).nonzero())]

def medoid_positions(labels):
    labels = pd.Series(np.asarray(labels).astype(str), index=ad.obs_names); pos = {}
    for k, idx in labels.groupby(labels).groups.items():
        p = pxy[ad.obs_names.get_indexer(idx)]; pos[k] = p[np.argmin(((p - p.mean(0)) ** 2).sum(1))]
    return labels, pos

def plot_method(col, name, labels=None, edges=None, graph_title=None):
    if col not in ad.obs: print(name, 'not run -> no plot'); return
    t = ad.obs[col].values
    labels, pos = medoid_positions(ad.obs['region'] if labels is None else labels)
    edges = PAGA_EDGES if edges is None else edges
    cl_t = pd.Series(t, index=labels.index).groupby(labels).mean()
    fig, ax = plt.subplots(1, 3, figsize=(21, 6.5), gridspec_kw={'width_ratios': [1, 1, 1.1]})
    # (a) pseudotime on H&E
    show_tissue(ax[0]); s = ax[0].scatter(pxy[:, 0], pxy[:, 1], c=t, cmap='viridis', s=9, vmin=0, vmax=1, alpha=.9)
    ax[0].scatter(*pxy[root], c='cyan', s=220, marker='*', edgecolors='k'); plt.colorbar(s, ax=ax[0], fraction=.04, label='pseudotime')
    ax[0].set_title(f'{name}: pseudotime (0 = root, 1 = latest)')
    # (b) trajectory graph: nodes = clusters at their tissue position, arrows from earlier to later cluster
    show_tissue(ax[1], .45); ax[1].scatter(pxy[:, 0], pxy[:, 1], c=t, cmap='viridis', s=4, vmin=0, vmax=1, alpha=.25)
    for a, b in edges:
        a, b = str(a), str(b)
        if a not in pos or b not in pos: continue
        if cl_t[a] > cl_t[b]: a, b = b, a
        ax[1].add_patch(FancyArrowPatch(pos[a], pos[b], arrowstyle='-|>', mutation_scale=18, lw=2.2, color='k', shrinkA=12, shrinkB=12))
    sizes = labels.value_counts()
    ax[1].scatter([pos[k][0] for k in cl_t.index], [pos[k][1] for k in cl_t.index], c=cl_t.values, cmap='viridis', vmin=0, vmax=1,
                  s=[60 + 3 * sizes[k] ** .8 for k in cl_t.index], edgecolors='k', zorder=3)
    for k in cl_t.index: ax[1].text(*pos[k], k, ha='center', va='center', fontsize=8, color='w', weight='bold', zorder=4)
    ax[1].set_title(graph_title or f'{name}: trajectory graph (region clusters, arrows early → late)')
    # (c) pseudotime per region, regions sorted by their mean
    reg_order = ad.obs.groupby('region', observed=True)[col].mean().sort_values().index
    data = [ad.obs.loc[ad.obs['region'] == r, col].values for r in reg_order]
    ax[2].boxplot(data, vert=False, showfliers=False); ax[2].set_yticks(range(1, len(reg_order) + 1))
    ax[2].set_yticklabels([f"{r}{'  (root)' if r == ROOT_CLUSTER else ''}" for r in reg_order])
    ax[2].set_xlabel('pseudotime'); ax[2].set_title('pseudotime per region (sorted)')
    for a in ax[:2]: a.axis('off')
    plt.tight_layout(); plt.savefig(f'{OUT}/fig_{name}.png', dpi=110, bbox_inches='tight'); plt.show()

# SpaceFlow & SpaTrack build no cluster graph of their own -> shared expression (PAGA) graph between regions, directed by their pseudotime
plot_method('t_spaceflow', 'SpaceFlow', graph_title='SpaceFlow: region graph (PAGA) directed by SpaceFlow pseudotime')
plot_method('t_spatrack',  'SpaTrack',  graph_title='SpaTrack: region graph (PAGA) directed by SpaTrack pseudotime')

# StaVia: its own cluster graph (edgelist_maxout over its own clusters)
try:
    plot_method('t_stavia', 'StaVia', labels=np.asarray(v.labels), edges=[tuple(e) for e in v.edgelist_maxout],
                graph_title="StaVia: its own cluster graph (clusters from StaVia)")
except Exception as e:
    print('StaVia own graph unavailable (', repr(e)[:80], ') -> shared graph'); plot_method('t_stavia', 'StaVia')

# stLearn: its PSTS cluster path (from the stLearn run) over its own Louvain clusters
try:
    info = json.load(open(f'{OUT}/raw_stlearn_info.json')); path = info.get('psts_path', [])
    plot_method('t_stlearn', 'stLearn', labels=ad.obs['stlearn_cluster'].values, edges=list(zip(path[:-1], path[1:])) or None,
                graph_title=f"stLearn: PSTS path {' → '.join(path)}" if path else None)
except Exception as e:
    print('stLearn path unavailable (', repr(e)[:80], ')'); plot_method('t_stlearn', 'stLearn')

# CASCAT: its causal cluster tree
try:
    cres = sc.read_h5ad(f'{OUT}/cascat_result.h5ad')   # v3.1d: per-dataset copy
    T = cres.uns['cascat_connectivities']; T = pd.DataFrame(T) if not isinstance(T, pd.DataFrame) else T
    tree_edges = [(str(T.index[i]), str(T.columns[j])) for i, j in zip(*np.nonzero(T.values))]
    plot_method('t_cascat', 'CASCAT', labels=ad.obs['cascat_cluster'].values, edges=tree_edges,
                graph_title='CASCAT: its causal cluster tree (clusters from CASCAT)')
except Exception as e:
    print('CASCAT tree unavailable (', repr(e)[:80], ')'); plot_method('t_cascat', 'CASCAT')


## 8. SPACECONS-R (consensus pseudotime) — the graph part here still uses the shared Leiden regions and is kept only for comparison; section 9 replaces it

## 8. SPACECONS — one consensus pseudotime and one trajectory graph from all five methods
Follows the project brief (steps 2, 3, 5, 6):
1. **Canonical form + orientation** — every method as rank-scaled pseudotime 0–1, root = 0 (flipped if needed).
2. **Reliability weights** `w ∝ S · C · (1 − SRI)`:
   * **S** self-reproducibility — *proxy*: how well each spot's pseudotime is reproduced by its expression neighbours (full bootstrap re-runs of every method are a later step);
   * **C** spatial coherence — Moran's I of the pseudotime on the spatial neighbour graph;
   * **SRI** spatial redundancy — R² of a smooth thin-plate-spline field of the coordinates fitted to the pseudotime (smoothness chosen by cross-validation). High SRI = "just geometry", so it lowers the weight.
   * `SHRINK = 0.5` mixes these weights 50/50 with equal weights so that no single method dominates (set 0 for the brief's formula only).
3. **SPACECONS-R (ordering)** — **weighted Borda** (weighted mean rank) per spot = the consensus pseudotime; checked against **uniform Borda**, **Kemeny** (exact, integer programming at region level), **MC4** (Markov-chain aggregation) and **Robust Rank Aggregation** (which spots are *consistently* early/late across methods).
4. **Uncertainty** — bootstrap over methods (500×): per-spot 95% interval and **local ordering confidence** |2π−1| with spatial neighbours; shown as a **confidence-masked map**.
5. **SPACECONS-G (trajectory graph)** — every method's graph (its own where it has one, else the shared PAGA graph tuned by its ordering) is reduced to a tree on the shared milestones (regions); **majority-rule consensus** gives each edge a support value; edges are directed by the consensus pseudotime.
   **Topology Disagreement Index** (mean pairwise Hamming–Ipsen–Mikhailov distance): if TDI > 0.5 the notebook **abstains** — reports the ordering and flags the topology as unresolved.
6. **Evaluation** — consensus vs each method, uniform Borda and the "space only" baseline (1st principal axis of the coordinates); SRI of the consensus.
Outputs: `spacecons_pseudotime.csv`, `spacecons_weights.csv`, `spacecons_graph_edges.csv`, `spacecons_evaluation.csv`, `spacecons_summary.json`, `fig_SPACECONS.png`.

### SPACECONS 2.1 weighting — correlated methods without a knife edge
Each method's quality (leave-one-out agreement × self-consistency) is divided by its *redundancy*: one plus a soft count of how many other methods correlate with it above TAU = 0.80. A smooth count is used because the 2.0 hard threshold changed the weights abruptly (on TBI, SpaceFlow–StaVia ρ = 0.8007 decides whether three methods form one family). The 2.0 family rule, the 1.x LOO² rule and equal weights are all scored in the validation table as ablations. None of these rules can tell a *shared error* from a *shared truth*; that needs independent references or simulations.

The milestone stage also exports a supported non-tree network (`spacecons_milestone_network_v2.csv`); the edge support is an ensemble-support score, not a causal probability. It is still built from each method's spanning tree, so it relaxes only the *consensus* tree.


In [ ]:
# [CELL:consensus]
# =====================  SPACECONS consensus: one pseudotime + one trajectory graph from all methods  =====================
import json, itertools, networkx as nx, scanpy as sc
from scipy.stats import rankdata, beta
from scipy.interpolate import RBFInterpolator
from scipy.optimize import milp, LinearConstraint, Bounds
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import KFold
rng = np.random.default_rng(0)

METH = [(c, n) for c, n in [('t_spaceflow', 'SpaceFlow'), ('t_spatrack', 'SpaTrack'), ('t_stavia', 'StaVia'),
                            ('t_stlearn', 'stLearn'), ('t_cascat', 'CASCAT')] if c in ad.obs]
cols = [c for c, _ in METH]; names = [n for _, n in METH]
T = np.column_stack([rank01(ad.obs[c].values) for c in cols])          # canonical form: rank-scaled 0..1, oriented root = 0
n, M = T.shape
print(f'{M} methods combined: {", ".join(names)}')

# ---------- Step 2: orientation check (every method must start at the root) ----------
for j, nm in enumerate(names):
    if T[(ad.obs['region'] == ROOT_CLUSTER).values, j].mean() > 0.5: T[:, j] = 1 - T[:, j]; print(f'  {nm} flipped so that the root is early')   # judged on the whole root region, not one spot

# ---------- Weights  w_m ∝ S_m · C_m · (1 − SRI_m)  (brief, step 3) ----------
# S: self-reproducibility. True bootstrap reruns of every method are expensive (step 6); here a cheap proxy:
#    how well each spot's pseudotime is reproduced by its expression neighbours (leave-self-out kNN average on PCA).
# C: spatial coherence = Moran's I of the pseudotime on the spatial kNN graph.
# SRI: R² of a smooth spatial field f(z) (thin-plate spline, smoothing chosen by 5-fold CV) fitted to the pseudotime.
Xp = ad.obsm['X_pca'][:, :30]
ie = NearestNeighbors(n_neighbors=16).fit(Xp).kneighbors(Xp, return_distance=False)[:, 1:]
isp = NearestNeighbors(n_neighbors=7).fit(xy).kneighbors(xy, return_distance=False)[:, 1:]
def morans_I(t, nb):
    z = t - t.mean(); return (z[:, None] * z[nb]).sum() / nb.size / (z ** 2).mean()
Z = (xy - xy.mean(0)) / xy.std(0)
def sri(t, smooth_grid=(1e-3, 1e-2, 1e-1, 1, 10)):
    best, best_s = -np.inf, None
    for s in smooth_grid:                                                   # choose smoothness by cross-validation (GCV stand-in)
        r2 = []
        for tr, te in KFold(5, shuffle=True, random_state=0).split(Z):
            f = RBFInterpolator(Z[tr], t[tr], kernel='thin_plate_spline', smoothing=s, neighbors=200)
            r2.append(1 - ((t[te] - f(Z[te])) ** 2).sum() / ((t[te] - t[te].mean()) ** 2).sum())
        if np.mean(r2) > best: best, best_s = np.mean(r2), s
    return float(np.clip(best, 0, 1)), best_s          # out-of-fold R² (in-sample R² is ~1 for any field once the spline nearly interpolates)
W = []
for j, nm in enumerate(names):
    t = T[:, j]
    S = max(spearmanr(t, t[ie].mean(1))[0], 0)
    C = max(morans_I(t, isp), 0)
    R, s = sri(t)
    W.append((nm, S, C, R, s))
Wt = pd.DataFrame(W, columns=['method', 'S (self-consistency)', 'C (Moran I)', 'SRI', 'TPS smoothing']).set_index('method')
Wt['raw weight S·C·(1−SRI)'] = Wt['S (self-consistency)'] * Wt['C (Moran I)'] * (1 - Wt['SRI'])
Wt['brief weight'] = Wt['raw weight S·C·(1−SRI)'] / Wt['raw weight S·C·(1−SRI)'].sum()
# SPACECONS 2.1: smooth redundancy-aware weighting (replaces the 2.0 hard |rho| >= 0.80 families).
# Why: with a hard threshold a method pair at rho = 0.801 is "one family" and at 0.799 "independent", and the weights
#      jump (TBI SpaceFlow–StaVia is 0.8007). Here every pair counts as a soft copy:
#        s(rho) = 1 / (1 + exp(-(rho - TAU) / TAU_WIDTH))   (~0 well below TAU, 0.5 at TAU, ~1 well above)
#        redundancy r_j = 1 + sum_k s(rho_jk)                (1 = unique method, ~3 = two near-copies)
#        quality    q_j = LOO agreement x (0.5 + 0.5 x self-consistency S)
#        weight     w_j ∝ q_j / r_j
# Reference labels are NOT used here. WEIGHTING picks the primary rule; every rule is still scored in validation.
from scipy.stats import spearmanr
WEIGHTING = 'loo_squared'   # primary, pre-specified rule used in the report; alternatives: 'redundancy_smooth' | 'family_threshold' | 'equal'
TAU, TAU_WIDTH = 0.80, 0.05
corr = np.eye(M)
for a in range(M):
    for b in range(a + 1, M):
        rr = spearmanr(T[:, a], T[:, b])[0]
        corr[a, b] = corr[b, a] = 0.0 if not np.isfinite(rr) else rr

loo = np.array([max(float(spearmanr(T[:, j], np.delete(T, j, axis=1).mean(1))[0]), 0.0) if M > 1 else 1.0
                for j in range(M)])
loo_only = loo ** 2 / (loo ** 2).sum() if (loo ** 2).sum() > 0 else np.ones(M) / M
quality = np.maximum(loo, 0.05) * (0.5 + 0.5 * np.clip(Wt['S (self-consistency)'].values, 0, 1))


def smooth_weights(tau, width=TAU_WIDTH):
    s_ = 1 / (1 + np.exp(-(corr - tau) / width)); np.fill_diagonal(s_, 1.0)
    r_ = s_.sum(1); wt = quality / r_
    return wt / wt.sum(), r_


def family_weights(threshold):
    """SPACECONS 2.0 rule, kept for the ablation: hard families at |rho| >= threshold."""
    Gt = nx.Graph(); Gt.add_nodes_from(range(M))
    Gt.add_edges_from((a, b) for a in range(M) for b in range(a + 1, M) if abs(corr[a, b]) >= threshold)
    groups_t = [np.array(sorted(g), dtype=int) for g in nx.connected_components(Gt)]
    rel = np.array([quality[g].mean() for g in groups_t]); mass = rel / rel.sum()
    wt = np.zeros(M)
    for gi, g in enumerate(groups_t):
        wt[g] = quality[g] / quality[g].sum() * mass[gi]
    return wt, groups_t


w_smooth, redundancy = smooth_weights(TAU)
w_family, method_groups = family_weights(TAU)
RULES = {'equal': np.ones(M) / M, 'loo_squared': loo_only, 'family_threshold': w_family, 'redundancy_smooth': w_smooth}
w = RULES[WEIGHTING]
_edge = [(names[a], names[b], round(corr[a, b], 3)) for a in range(M) for b in range(a + 1, M) if abs(abs(corr[a, b]) - TAU) < 0.02]
if _edge:
    print(f'NOTE: method pairs within 0.02 of TAU = {TAU} (the 2.0 family rule is unstable for these):', _edge)

Wt['LOO agreement'] = loo
Wt['redundancy r'] = redundancy
Wt['2.0 family'] = [next(f'G{gi + 1}' for gi, g in enumerate(method_groups) if j in g) for j in range(M)]
Wt['equal weight'] = RULES['equal']
Wt['LOO-only weight'] = loo_only
Wt['family weight (2.0)'] = w_family
Wt['smooth weight (2.1)'] = w_smooth
Wt['weight'] = w  # downstream cells use the primary rule
print(f'\nPrimary weighting rule: {WEIGHTING}  (TAU = {TAU})')
display(Wt.round(3))
pd.DataFrame(corr, index=names, columns=names).to_csv(f'{OUT}/method_spearman_correlation.csv')
Wt.round(4).to_csv(f'{OUT}/spacecons_weights_v2.csv')

# Ablation (scored against the reference ONLY in the validation cell; never used to choose weights here)
weight_ablation = pd.DataFrame(RULES, index=names)
threshold_weight_sensitivity = pd.concat(
    [pd.Series(smooth_weights(t)[0], index=names, name=f'smooth tau {t:.2f}') for t in (0.70, 0.80, 0.90)] +
    [pd.Series(family_weights(t)[0], index=names, name=f'family tau {t:.2f}') for t in (0.70, 0.80, 0.90)], axis=1)
threshold_weight_sensitivity.to_csv(f'{OUT}/spacecons_weight_threshold_sensitivity.csv')
print('Weight sensitivity to TAU (rows = methods):'); display(threshold_weight_sensitivity.round(3))

# ---------- SPACECONS-R: consensus ordering ----------
borda_w = rank01((T * w).sum(1))                       # weighted Borda = weighted mean rank (primary estimator)
borda_u = rank01(T.mean(1))                            # uniform Borda (baseline the weighting must beat)
# Robust Rank Aggregation (Kolde 2012): is a spot consistently EARLY (or LATE) across methods?
def rra_p(R):
    Rs = np.sort(R, 1); k = np.arange(1, R.shape[1] + 1)
    return np.minimum(beta.cdf(Rs, k, R.shape[1] - k + 1).min(1) * R.shape[1], 1)
p_early, p_late = rra_p(T), rra_p(1 - T)
# milestone level (regions) for Kemeny and MC4
reg = ad.obs['region'].astype(str).values; K_lab = sorted(set(reg), key=lambda s: int(s) if s.isdigit() else s)
K = len(K_lab); Tm = np.array([[T[reg == k, j].mean() for j in range(M)] for k in K_lab])    # K x M milestone pseudotimes
P = np.zeros((K, K))                                   # P[a,b] = weighted share of methods putting milestone a before b
for j in range(M): P += w[j] * (Tm[:, j][:, None] < Tm[:, j][None, :])
# Kemeny (exact, integer programming over pairwise orders with transitivity constraints)
idx = {(a, b): i for i, (a, b) in enumerate((a, b) for a in range(K) for b in range(K) if a != b)}
c = np.zeros(len(idx)); A, lb, ub = [], [], []
for (a, b), i in idx.items(): c[i] = -P[a, b]
for a in range(K):
    for b in range(a + 1, K):
        row = np.zeros(len(idx)); row[idx[a, b]] = row[idx[b, a]] = 1; A.append(row); lb.append(1); ub.append(1)
for a, b, d in itertools.permutations(range(K), 3):
    row = np.zeros(len(idx)); row[idx[a, b]] = row[idx[b, d]] = row[idx[d, a]] = 1; A.append(row); lb.append(0); ub.append(2)
sol = milp(c, constraints=LinearConstraint(np.array(A), lb, ub), integrality=np.ones(len(idx)), bounds=Bounds(0, 1))
wins = np.zeros(K)
for (a, b), i in idx.items(): wins[a] += round(sol.x[i]) if sol.success else P[a, b]
kemeny_order = [K_lab[i] for i in np.argsort(-wins)]                                     # earliest first
# MC4 (Dwork et al. 2001): random walk that moves from a milestone to one ranked earlier by a weighted majority
Q = np.where(P.T > 0.5, 1.0, 0.0) / K; np.fill_diagonal(Q, 0); np.fill_diagonal(Q, 1 - Q.sum(1))
Q = 0.95 * Q + 0.05 / K                                # small random jump: otherwise all mass sinks into the earliest milestone and the rest tie
ev, evec = np.linalg.eig(Q.T); pi = np.abs(np.real(evec[:, np.argmin(np.abs(ev - 1))])); pi /= pi.sum()
mc4_order = [K_lab[i] for i in np.argsort(-pi)]                                          # most visited = earliest
borda_order = [K_lab[i] for i in np.argsort(Tm @ w)]
print('Milestone (region) order, earliest -> latest:')
print('  weighted Borda:', ' '.join(borda_order)); print('  Kemeny (exact):', ' '.join(kemeny_order)); print('  MC4           :', ' '.join(mc4_order))
print(f"  rank agreement Borda vs Kemeny (Spearman) = {spearmanr([borda_order.index(k) for k in K_lab], [kemeny_order.index(k) for k in K_lab])[0]:.3f} (Spearman)")

ad.obs['t_spacecons'] = borda_w; ad.obs['t_spacecons_uniform'] = borda_u
ad.obs['rra_p_early'] = p_early; ad.obs['rra_p_late'] = p_late

# ---------- Uncertainty (method bootstrap): 95% interval per spot + local ordering confidence ----------
B = 500; boot = np.empty((B, n))
# Method bootstrap: draw M methods with replacement and use their (already redundancy-discounted) weights.
# 2.1 fix: the 2.0 version drew one method per family but kept that method's within-family share of the weight,
# so a 3-method family counted about 1/3 of its intended mass, and a single family gave single-method replicates.
for b in range(B):
    pick = rng.integers(0, M, M)
    ww = w[pick]; ww = ww / ww.sum() if ww.sum() > 0 else np.ones(M) / M
    boot[b] = rank01((T[:, pick] * ww).sum(1))
lo, hi = np.percentile(boot, [2.5, 97.5], 0); ad.obs['spacecons_ci_width'] = hi - lo
conf = np.array([np.mean(np.abs(2 * (boot[:, [i]] < boot[:, isp[i]]).mean(0) - 1)) for i in range(n)])   # |2·π_ij − 1| over spatial neighbours
ad.obs['spacecons_local_conf'] = conf
print(f'\nUncertainty: median 95% interval width = {np.median(hi - lo):.3f}; median local ordering confidence = {np.median(conf):.2f}')
print('Spots consistently EARLY in all methods (RRA p<0.05):', int((p_early < .05).sum()), '| consistently LATE:', int((p_late < .05).sum()))

# ---------- SPACECONS-G: consensus trajectory graph (majority rule over every method's tree on shared milestones) ----------
def majority_region(lbl):
    s = pd.Series(reg, index=ad.obs_names).groupby(pd.Series(np.asarray(lbl).astype(str), index=ad.obs_names))
    return s.agg(lambda x: x.value_counts().index[0]).to_dict()
sc.tl.paga(ad, groups='region'); Cp = ad.uns['paga']['connectivities'].toarray()
cats = list(ad.obs['region'].cat.categories)
paga_ew = {(cats[i], cats[j]): Cp[i, j] for i, j in zip(*np.triu(Cp > 0.05, 1).nonzero())}
native = {}
try: native['StaVia'] = (np.asarray(v.labels), [tuple(e) for e in v.edgelist_maxout])
except Exception: pass
try:
    pth = json.load(open(f'{OUT}/raw_stlearn_info.json')).get('psts_path', [])
    native['stLearn'] = (ad.obs['stlearn_cluster'].values, list(zip(pth[:-1], pth[1:])))
except Exception: pass
try:
    cr = sc.read_h5ad(f'{OUT}/cascat_result.h5ad')   # v3.1d: per-dataset copy; Tc = cr.uns['cascat_connectivities']
    Tc = Tc if isinstance(Tc, pd.DataFrame) else pd.DataFrame(Tc)
    native['CASCAT'] = (ad.obs['cascat_cluster'].values, [(str(Tc.index[i]), str(Tc.columns[j])) for i, j in zip(*np.nonzero(Tc.values))])
except Exception: pass
root_reg = str(ad.obs['region'].iloc[root]); trees = {}
for j, nm in enumerate(names):
    G = nx.Graph(); G.add_nodes_from(K_lab)
    for (a, b), cval in paga_ew.items(): G.add_edge(a, b, weight=cval)              # shared expression backbone (weight < 1)
    if nm in native:                                                                    # method's own graph, mapped to regions
        lbl, edges = native[nm]; mp = majority_region(lbl)
        for a, b in edges:
            ra, rb = mp.get(str(a)), mp.get(str(b))
            if ra and rb and ra != rb: G.add_edge(ra, rb, weight=G.get_edge_data(ra, rb, {'weight': 0})['weight'] + 2)
    else:                                                                               # no own graph: edges agreeing with its ordering get a bonus
        for a, b in list(G.edges()):
            G[a][b]['weight'] += 1 - abs(Tm[K_lab.index(a), j] - Tm[K_lab.index(b), j])
    if not nx.is_connected(G):
        comps = list(nx.connected_components(G))
        for c1, c2 in zip(comps[:-1], comps[1:]): G.add_edge(next(iter(c1)), next(iter(c2)), weight=1e-3)
    trees[nm] = nx.maximum_spanning_tree(G)
support = {}
for j, nm in enumerate(names):
    for a, b in trees[nm].edges(): support[frozenset((a, b))] = support.get(frozenset((a, b)), 0) + w[j]
Gs = nx.Graph(); Gs.add_nodes_from(K_lab)
for e, s in support.items(): Gs.add_edge(*tuple(e), support=s)
cons_tree = nx.maximum_spanning_tree(Gs, weight='support')                            # majority edges first; low-support edges only to connect
cl_t = pd.Series(borda_w, index=ad.obs_names).groupby(reg).mean()
cons_dir = [(a, b) if cl_t[a] <= cl_t[b] else (b, a) for a, b in cons_tree.edges()]
# Topology Disagreement Index = mean pairwise Hamming–Ipsen–Mikhailov distance between the method trees
def him(G1, G2, xi=1.0, gamma=0.08):
    A1 = nx.to_numpy_array(G1, nodelist=K_lab) > 0; A2 = nx.to_numpy_array(G2, nodelist=K_lab) > 0
    H = (A1 != A2).sum() / (K * (K - 1))
    def dens(A):
        om = np.sqrt(np.clip(np.linalg.eigvalsh(np.diag(A.sum(1)) - A.astype(float)), 0, None))[1:]
        x = np.linspace(0, om.max() + 3, 600); r = (gamma / ((x[:, None] - om[None]) ** 2 + gamma ** 2)).sum(1)
        return x, r / np.trapezoid(r, x)
    (x1, r1), (x2, r2) = dens(A1), dens(A2); x = np.linspace(0, max(x1.max(), x2.max()), 800)
    IM = np.sqrt(np.trapezoid((np.interp(x, x1, r1) - np.interp(x, x2, r2)) ** 2, x))
    return float(np.sqrt(H ** 2 / (1 + xi) + xi * IM ** 2 / (1 + xi)))
pairs = list(itertools.combinations(names, 2)); TDI = float(np.mean([him(trees[a], trees[b]) for a, b in pairs])) if pairs else 0.0
TDI_THRESHOLD = 0.5
topo_flag = 'topology RESOLVED' if TDI <= TDI_THRESHOLD else 'topology UNRESOLVED (methods disagree): use the ordering, treat the graph as tentative'
print(f'\nTopology Disagreement Index (mean pairwise HIM) = {TDI:.3f} -> {topo_flag}')
edge_tab = pd.DataFrame([(a, b, support.get(frozenset((a, b)), 0)) for a, b in cons_dir], columns=['from (earlier)', 'to (later)', 'support'])
print('Consensus trajectory edges (support = weighted share of methods whose tree contains the edge):')
display(edge_tab.sort_values('support', ascending=False).round(2))

# ---------- How good is the consensus? ----------
inj = ad.obs['score_injury'].values
rows = [(nm, spearmanr(T[:, j], borda_w)[0], spearmanr(T[:, j], inj)[0], T[ad.obs['region'].values == ROOT_CLUSTER, j].mean()) for j, nm in enumerate(names)]
rows += [('SPACECONS (weighted)', 1.0, spearmanr(borda_w, inj)[0], borda_w[ad.obs['region'].values == ROOT_CLUSTER].mean()),
         ('Borda (uniform)', spearmanr(borda_u, borda_w)[0], spearmanr(borda_u, inj)[0], borda_u[ad.obs['region'].values == ROOT_CLUSTER].mean()),
         ('space only (1st PC of xy)', abs(spearmanr(np.linalg.svd(Z, full_matrices=False)[0][:, 0], borda_w)[0]), np.nan, np.nan)]
ev_tab = pd.DataFrame(rows, columns=['ordering', 'agreement with SPACECONS', f'corr. with {TARGET_NAME}', 'root-cluster mean']).set_index('ordering')
print('\nConsensus vs individual methods and baselines:'); display(ev_tab.round(3))
cons_sri, _ = sri(borda_w); print(f'SRI of the consensus = {cons_sri:.3f}  (1 = recoverable from position alone; tested against null models in the next step)')

# ---------- Figures ----------
img, pxy = tissue_image()
pos = {}
for k in K_lab:
    p = pxy[reg == k]; pos[k] = p[np.argmin(((p - p.mean(0)) ** 2).sum(1))]
fig, ax = plt.subplots(1, 4, figsize=(28, 7))
for a in ax: show_tissue(a, .55); a.axis('off')
s = ax[0].scatter(*pxy.T, c=borda_w, cmap='viridis', s=9, vmin=0, vmax=1); plt.colorbar(s, ax=ax[0], fraction=.04)
ax[0].scatter(*pxy[root], c='cyan', s=220, marker='*', edgecolors='k'); ax[0].set_title('SPACECONS consensus pseudotime')
alpha = np.clip(conf, 0.05, 1); s = ax[1].scatter(*pxy.T, c=borda_w, cmap='viridis', s=9, vmin=0, vmax=1, alpha=alpha)
ax[1].set_title('confidence-masked (faded = methods disagree locally)')
s = ax[2].scatter(*pxy.T, c=hi - lo, cmap='magma', s=9); plt.colorbar(s, ax=ax[2], fraction=.04); ax[2].set_title('95% bootstrap interval width')
from matplotlib.patches import FancyArrowPatch
for a_, b_ in cons_dir:
    sup = support.get(frozenset((a_, b_)), 0)
    ax[3].add_patch(FancyArrowPatch(pos[a_], pos[b_], arrowstyle='-|>', mutation_scale=18, lw=1 + 4 * sup,
                                    color='k' if sup > 0.5 else 'grey', linestyle='-' if sup > 0.5 else '--', shrinkA=12, shrinkB=12))
ax[3].scatter([pos[k][0] for k in K_lab], [pos[k][1] for k in K_lab], c=[cl_t[k] for k in K_lab], cmap='viridis', vmin=0, vmax=1,
              s=[60 + 2 * (reg == k).sum() ** .8 for k in K_lab], edgecolors='k', zorder=3)
for k in K_lab: ax[3].text(*pos[k], k + ('*' if k == root_reg else ''), ha='center', va='center', fontsize=8, color='w', weight='bold', zorder=4)
ax[3].set_title(f'SPACECONS trajectory graph (solid = majority support)\nTDI {TDI:.2f}: {topo_flag.split(" (")[0]}')
plt.tight_layout(); plt.savefig(f'{OUT}/fig_SPACECONS.png', dpi=110, bbox_inches='tight'); plt.show()

Wt.round(4).to_csv(f'{OUT}/spacecons_weights.csv'); ev_tab.round(4).to_csv(f'{OUT}/spacecons_evaluation.csv')
edge_tab.to_csv(f'{OUT}/spacecons_graph_edges.csv', index=False)
pd.DataFrame({'barcode': ad.obs_names, 'x': xy[:, 0], 'y': xy[:, 1], 'region': reg, 'spacecons_pseudotime': borda_w,
              'ci_low': lo, 'ci_high': hi, 'local_confidence': conf, 'rra_p_early': p_early, 'rra_p_late': p_late}
             ).to_csv(f'{OUT}/spacecons_pseudotime.csv', index=False)
json.dump({'milestone_order_borda': borda_order, 'milestone_order_kemeny': kemeny_order, 'milestone_order_mc4': mc4_order,
           'TDI': TDI, 'topology': topo_flag, 'consensus_SRI': cons_sri}, open(f'{OUT}/spacecons_summary.json', 'w'), indent=1)
print('Saved spacecons_*.csv, spacecons_summary.json, fig_SPACECONS.png')


## 9. SPACECONS-M — consensus milestones and milestone graph from each method's own milestones
Each method has its own milestones (its own clusters) and its own graph, with different numbers of milestones. They are compared **through the spots**:
1. per method: its graph → one milestone tree; milestones ordered by that method's pseudotime from the root;
2. **consensus milestones** — weighted co-membership (share of methods that put two spots in the same milestone) → average-linkage tree; the number of milestones = weighted median of the methods' own milestone counts (their typical level of detail), so it comes from the methods, not from us;
3. **consensus graph** — each method's edges are transferred onto the consensus milestones in proportion to the spots they share; best-supported tree, directed by the SPACECONS-R pseudotime;
4. **consensus milestone ordering** = consensus milestones sorted by consensus pseudotime, with the majority reference state of each.

In [ ]:
# [CELL:milestones]
# =====================  SPACECONS-M: consensus milestones + milestone graph from each method's OWN milestones  =====================
# Each method keeps its own milestones (its own clusters) and its own graph; the methods are compared THROUGH THE SPOTS:
#   1. per method: milestone tree (its graph, made a tree), milestones ordered by that method's pseudotime from the root
#   2. consensus milestones: co-membership matrix (weighted share of methods that put two spots in the same milestone),
#      average-linkage tree; the NUMBER of milestones = weighted median of the methods' own counts (not fixed by us)
#   3. consensus graph: every method's edges are transferred onto the consensus milestones in proportion to shared spots
#   4. consensus milestone ordering = consensus milestones sorted by the SPACECONS-R pseudotime (weighted Borda)
import json, networkx as nx
from collections import defaultdict
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from scipy.integrate import quad
from scipy.linalg import eigh
from scipy.sparse.csgraph import laplacian

MCOL = {'SpaceFlow': 'm_spaceflow', 'SpaTrack': 'm_spatrack', 'StaVia': 'm_stavia', 'stLearn': 'm_stlearn', 'CASCAT': 'm_cascat'}
json.dump({k: [[str(a), str(b), float(c)] for a, b, c in v_] for k, v_ in MS_GRAPH.items()}, open(f'{OUT}/method_graphs.json', 'w'))
use = [(j, nm) for j, nm in enumerate(names) if MCOL.get(nm) in ad.obs and nm in MS_GRAPH]
print('methods with their own milestones + graph:', ', '.join(nm for _, nm in use))
n = ad.n_obs
ref = ad.obs['ref_state'].astype(object).where(ad.obs['ref_state'].notna(), '-').values

def ref_major(mask):                                     # majority reference state of a group of spots (for labelling)
    s = pd.Series(ref[mask]); s = s[s != '-']
    return s.value_counts().index[0] if len(s) else '-'

def as_tree(lab, edges, t):
    """the method's milestone graph -> one tree (max spanning tree on its edge weights); separate pieces are joined by the
    milestones closest in pseudotime (tiny weight), so every method yields one connected milestone tree."""
    ms = sorted(set(lab)); tm = pd.Series(t).groupby(lab).mean()
    G = nx.Graph(); G.add_nodes_from(ms)
    for a, b, c in edges:
        a, b = str(a), str(b)
        if a in G and b in G and a != b:
            G.add_edge(a, b, weight=max(float(c), G.get_edge_data(a, b, {'weight': 0})['weight']))
    comps = [list(c) for c in nx.connected_components(G)]
    while len(comps) > 1:
        best = min(((abs(tm[a] - tm[b]), a, b) for a in comps[0] for c in comps[1:] for b in c))
        G.add_edge(best[1], best[2], weight=1e-6, bridge=True)
        comps = [list(c) for c in nx.connected_components(G)]
    return nx.maximum_spanning_tree(G, weight='weight'), tm

trees, labs, mt = {}, {}, {}
rows = []
for j, nm in use:
    lab = ad.obs[MCOL[nm]].astype(str).values
    tree, tm = as_tree(lab, MS_GRAPH[nm], T[:, j])
    trees[nm], labs[nm], mt[nm] = tree, lab, tm
    order = tm.sort_values().index.tolist()
    rows.append({'method': nm, 'milestones': len(order), 'root milestone': lab[root],
                 'ordering (milestone[reference state])': ' → '.join(f'{m}[{ref_major(lab == m)}]' for m in order)})
MO = pd.DataFrame(rows).set_index('method')
print('\nEach method’s own milestone ordering (earliest → latest), with the majority reference state of each milestone:')
display(MO)

# ---- consensus milestones: weighted co-membership + average-linkage tree ----
wj = np.array([w[j] for j, _ in use]); wj = wj / wj.sum()
A = np.zeros((n, n), dtype=np.float32)
for (j, nm), wt in zip(use, wj):
    codes = pd.factorize(labs[nm])[0]
    A += wt * (codes[:, None] == codes[None, :])
D = 1.0 - A; np.fill_diagonal(D, 0)
Z = linkage(squareform(D, checks=False).astype(float), method='average')
# how many consensus milestones? the weighted MEDIAN of the methods' own milestone counts (the typical level of detail);
# the agreement (weighted adjusted mutual information) with the methods is printed for every candidate count as a check
from sklearn.metrics import adjusted_mutual_info_score
kc = pd.Series({nm: len(set(labs[nm])) for _, nm in use}); kw = pd.Series(wj, index=kc.index)
o = kc.sort_values().index; K_best = int(kc[o][(kw[o].cumsum() >= 0.5).values].iloc[0])
kscore = {k: sum(wt * adjusted_mutual_info_score(labs[nm], fcluster(Z, t=k, criterion='maxclust')) for (_, nm), wt in zip(use, wj))
          for k in range(max(2, kc.min()), kc.max() + 1)}
print(f'consensus milestone count = weighted median of the methods’ counts = {K_best} '
      f'(agreement with the methods, weighted AMI: {kscore[K_best]:.3f}; best possible in {kc.min()}–{kc.max()}: '
      f'{max(kscore.values()):.3f} at K = {max(kscore, key=kscore.get)})')
cm = fcluster(Z, t=K_best, criterion='maxclust')
MIN_SIZE = max(10, int(0.005 * n))                       # tiny groups are merged into the group they co-occur with most
while True:
    sizes = pd.Series(cm).value_counts(); small = sizes[sizes < MIN_SIZE].index
    if len(small) == 0 or len(sizes) <= 2: break
    k = small[0]; idx = np.where(cm == k)[0]
    others = [c for c in sizes.index if c != k]
    cm[idx] = others[int(np.argmax([A[np.ix_(idx, np.where(cm == c)[0])].mean() for c in others]))]
cmt = pd.Series(borda_w).groupby(cm).mean().sort_values()   # order consensus milestones by consensus pseudotime
rename = {c: f'C{i + 1}' for i, c in enumerate(cmt.index)}
CM = np.array([rename[c] for c in cm]); ad.obs['spacecons_milestone'] = CM
C_lab = [rename[c] for c in cmt.index]
del A, D
_counts = ', '.join(f'{nm} {MO.loc[nm, "milestones"]}' for _, nm in use)
print(f'\nConsensus milestones: {len(C_lab)} (methods have {_counts})')

# ---- consensus graph: transfer every method's tree edges onto the consensus milestones by spot overlap ----
S = np.zeros((len(C_lab), len(C_lab)))
for (j, nm), wt in zip(use, wj):
    ms = sorted(set(labs[nm]))
    O = pd.crosstab(pd.Series(labs[nm]), pd.Series(CM)).reindex(index=ms, columns=C_lab, fill_value=0)
    O = O.div(O.sum(1), axis=0).values                     # O[k, a] = share of milestone k's spots that sit in consensus milestone a
    B = nx.to_numpy_array(trees[nm], nodelist=ms, weight=None)
    Mx = O.T @ B @ O; np.fill_diagonal(Mx, 0)              # edge (k,l) votes for (a,b) with weight O[k,a]·O[l,b] (+ reverse)
    S += wt * np.minimum(Mx, 1.0)
Gs = nx.Graph(); Gs.add_nodes_from(C_lab)
for a in range(len(C_lab)):
    for b in range(a + 1, len(C_lab)):
        if S[a, b] > 0: Gs.add_edge(C_lab[a], C_lab[b], support=float(S[a, b]))
ct = pd.Series(borda_w).groupby(CM).mean()

# SPACECONS 2.0 graph output: retain supported non-tree edges as well as the legacy tree.
# This allows branching/alternative paths to be inspected instead of forcing every result into a tree.
EDGE_SUPPORT_THRESHOLD = 0.25
network_rows = []
for a in range(len(C_lab)):
    for b in range(a + 1, len(C_lab)):
        support_ab = float(S[a, b])
        if support_ab >= EDGE_SUPPORT_THRESHOLD:
            src, dst = (C_lab[a], C_lab[b]) if ct[C_lab[a]] <= ct[C_lab[b]] else (C_lab[b], C_lab[a])
            network_rows.append((src, dst, support_ab))
consensus_network_edges = pd.DataFrame(
    network_rows, columns=['from (earlier)', 'to (later)', 'support']
).sort_values('support', ascending=False) if network_rows else pd.DataFrame(
    columns=['from (earlier)', 'to (later)', 'support']
)
consensus_network_edges.to_csv(f'{OUT}/spacecons_milestone_network_v2.csv', index=False)
print(f'SPACECONS 2.0 supported milestone network: {len(consensus_network_edges)} edges '
      f'(support >= {EDGE_SUPPORT_THRESHOLD:.2f}); legacy spanning tree retained below.')

comps = [list(c) for c in nx.connected_components(Gs)]
while len(comps) > 1:                                      # join pieces by the closest milestones in consensus pseudotime
    best = min(((abs(ct[a] - ct[b]), a, b) for a in comps[0] for c in comps[1:] for b in c))
    Gs.add_edge(best[1], best[2], support=0.0); comps = [list(c) for c in nx.connected_components(Gs)]
cons_mtree = nx.maximum_spanning_tree(Gs, weight='support')
cons_medges = sorted([((a, b) if ct[a] <= ct[b] else (b, a), Gs[a][b]['support']) for a, b in cons_mtree.edges()], key=lambda e: -e[1])
E = pd.DataFrame([(a, b, s_) for (a, b), s_ in cons_medges], columns=['from (earlier)', 'to (later)', 'support'])
comp = pd.DataFrame({'spots': pd.Series(CM).value_counts().reindex(C_lab).values,
                     'mean pseudotime': ct.reindex(C_lab).round(3).values,
                     'reference state (majority)': [ref_major(CM == c) for c in C_lab]}, index=C_lab)
print('\nConsensus milestone ordering (earliest → latest):'); display(comp)
print('Consensus milestone graph (support = weighted share of methods whose own edges connect the two milestones):'); display(E.round(2))

# ---- topology disagreement between the methods' own trees (Ipsen–Mikhailov, as in CASCAT; works across tree sizes) ----
def im_dist(G1, G2, hwhm=0.08):
    a1, a2 = nx.to_numpy_array(G1, weight=None), nx.to_numpy_array(G2, weight=None)
    w1 = np.sqrt(np.abs(eigh(laplacian(a1))[0][1:])); w2 = np.sqrt(np.abs(eigh(laplacian(a2))[0][1:]))
    n1 = (len(a1) - 1) * np.pi / 2 - np.sum(np.arctan(-w1 / hwhm)); n2 = (len(a2) - 1) * np.pi / 2 - np.sum(np.arctan(-w2 / hwhm))
    f = lambda x: (np.sum(hwhm / ((x - w1) ** 2 + hwhm ** 2)) / n1 - np.sum(hwhm / ((x - w2) ** 2 + hwhm ** 2)) / n2) ** 2
    return float(np.sqrt(quad(f, 0, np.inf, limit=100)[0]))
pairs = [(a, b) for i, a in enumerate(trees) for b in list(trees)[i + 1:]]
TDI_M = float(np.mean([im_dist(trees[a], trees[b]) for a, b in pairs])) if pairs else float('nan')
print(f'Topology disagreement between the methods’ own milestone trees (mean pairwise IM) = {TDI_M:.3f}')

# ---- figure: consensus milestones | consensus milestone graph | reference-state make-up of each consensus milestone ----
img, pxy = tissue_image()
mpos = {c: pxy[CM == c][np.argmin(((pxy[CM == c] - pxy[CM == c].mean(0)) ** 2).sum(1))] for c in C_lab}
from matplotlib.patches import FancyArrowPatch
fig, ax = plt.subplots(1, 3, figsize=(24, 7), gridspec_kw={'width_ratios': [1, 1, 0.9]})
for a_ in ax[:2]: show_tissue(a_, .55); a_.axis('off')
cmap = plt.get_cmap('tab20')
for i, c in enumerate(C_lab):
    m_ = CM == c; ax[0].scatter(*pxy[m_].T, s=8, color=cmap(i % 20), label=c)
ax[0].legend(fontsize=8, markerscale=2, ncol=2, loc='lower left'); ax[0].set_title(f'Consensus milestones ({len(C_lab)})')
for (a_, b_), s_ in cons_medges:
    ax[1].add_patch(FancyArrowPatch(mpos[a_], mpos[b_], arrowstyle='-|>', mutation_scale=18, lw=1 + 4 * s_,
                                    color='k' if s_ >= 0.5 else 'grey', linestyle='-' if s_ >= 0.5 else '--', shrinkA=12, shrinkB=12))
sc_ = ax[1].scatter([mpos[c][0] for c in C_lab], [mpos[c][1] for c in C_lab], c=[ct[c] for c in C_lab], cmap='viridis', vmin=0, vmax=1,
                    s=[60 + 2 * (CM == c).sum() ** .8 for c in C_lab], edgecolors='k', zorder=3)
for c in C_lab: ax[1].text(*mpos[c], c, ha='center', va='center', fontsize=8, color='w', weight='bold', zorder=4)
plt.colorbar(sc_, ax=ax[1], fraction=.04, label='mean consensus pseudotime')
ax[1].set_title('Consensus milestone graph (solid = majority support)')
mix = pd.crosstab(pd.Series(CM, name='milestone'), pd.Series(ref, name='ref')).reindex(C_lab)
mix = mix.drop(columns=[c for c in mix.columns if c == '-'], errors='ignore')
(mix.div(mix.sum(1).replace(0, 1), axis=0)).plot(kind='barh', stacked=True, ax=ax[2], colormap='coolwarm', width=.8)
ax[2].invert_yaxis(); ax[2].set_xlabel('share of reference-labelled spots'); ax[2].set_title('Reference state of each consensus milestone\n(top = earliest)')
plt.tight_layout(); plt.savefig(f'{OUT}/fig_SPACECONS_milestones.png', dpi=110, bbox_inches='tight'); plt.show()

MO.to_csv(f'{OUT}/method_milestone_orderings.csv'); comp.to_csv(f'{OUT}/spacecons_milestones.csv'); E.to_csv(f'{OUT}/spacecons_milestone_edges.csv', index=False)
pd.DataFrame({'barcode': ad.obs_names, 'spacecons_milestone': CM, 'spacecons_pseudotime': borda_w}).to_csv(f'{OUT}/spacecons_milestone_per_spot.csv', index=False)
print('Saved method_milestone_orderings.csv, spacecons_milestones.csv, spacecons_milestone_edges.csv, fig_SPACECONS_milestones.png')


## 10. Validation against the stLearn trajectory (CASCAT's metrics)
As CASCAT does on simulated data (`utils/Metrics.py`): **IM** and **OT** distance between the predicted milestone graph and the true milestone network (lower = better), **Kendall τ** and **Spearman ρ** between pseudotime and true time, **ARI / AMI** of milestones vs true labels (higher = better).
Here the "truth" is the trajectory stated by stLearn (chain of reference states; true time = a spot's position in that chain).

In [ ]:
# [CELL:validation]
# =====================  Validation against the trajectory stated by stLearn, with CASCAT's metrics  =====================
# CASCAT (utils/Metrics.py, caculate_metric) scores a method against a known trajectory with:
#   IM  Ipsen–Mikhailov distance between the predicted milestone graph and the true milestone network   (lower = better)
#   OT  Gromov–Wasserstein (optimal transport) distance between the same two graphs                     (lower = better)
#   KT  Kendall τ  between predicted pseudotime and true time                                           (higher = better)
#   SR  Spearman ρ between predicted pseudotime and true time                                           (higher = better)
#   ARI / AMI  predicted clusters (milestones) vs true labels                                           (higher = better)
# Here the "truth" is the trajectory stated by stLearn: true network = the chain of REF_STATES, true time = position of a
# spot's reference state in that chain (spots without a clear reference state are left out of KT / SR), true labels = ref_label.
# Extra column: does the method put the reference states in the stated order (mean pseudotime per state)?
from scipy.stats import kendalltau, rankdata, spearmanr
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score
from ot.gromov import gromov_wasserstein

def ot_dist(G1, G2):                                      # CASCAT's OTDistance
    a1 = np.where(nx.to_numpy_array(G1) > 0, 1.0, 0); a2 = np.where(nx.to_numpy_array(G2) > 0, 1.0, 0)
    h1, h2 = np.ones(len(a1)) / len(a1), np.ones(len(a2)) / len(a2)
    _, log = gromov_wasserstein(a1, a2, h1, h2, 'square_loss', symmetric=True, log=True, verbose=False, epsilon=5e-4, max_iter=1000)
    return float(log['gw_dist'])

REF_G = nx.Graph(); REF_G.add_nodes_from(REF_STATES); REF_G.add_edges_from(REF_EDGES)
rt = ad.obs['ref_time'].values.astype(float); ok = np.isfinite(rt)
rl = ad.obs['ref_label'].astype(str).values

# NEW baseline: distance from the root THROUGH the tissue (shortest path on the 6-nearest-neighbour spot graph).
# If a method only measures "how far from the root", it will not beat this baseline.
from scipy.sparse.csgraph import dijkstra
from scipy.sparse import csr_matrix
d_, i_ = NearestNeighbors(n_neighbors=7).fit(xy).kneighbors(xy)
Gk = csr_matrix((d_[:, 1:].ravel(), (np.repeat(np.arange(len(xy)), 6), i_[:, 1:].ravel())), shape=(len(xy), len(xy)))
DGEO = dijkstra(Gk, directed=False, indices=root)
DGEO[~np.isfinite(DGEO)] = np.nanmax(DGEO[np.isfinite(DGEO)])      # spots on a separate tissue piece -> farthest
DGEO = rank01(DGEO)

def partial_sr(pt):
    """Spearman of pseudotime with the reference time AFTER removing what distance-from-root explains (partial correlation on ranks).
    > 0: the method knows something about the trajectory beyond distance; ≈ 0: it is mostly distance."""
    a, b, c_ = (rankdata(v[ok]) for v in (pt, rt, DGEO))
    res = lambda y: y - np.polyval(np.polyfit(c_, y, 1), c_)
    return float(np.corrcoef(res(a), res(b))[0, 1])

def state_graph(tree, lab):
    """collapse a milestone tree to the reference states: each milestone takes the majority reference state of its spots;
    milestones with no reference-labelled spots are removed and their neighbours joined; an edge between two
    different states is kept. Result: which reference states the method connects directly."""
    rs = ad.obs['ref_state'].astype(object).values; lab = np.asarray(lab).astype(str)
    maj = {}
    for m in tree.nodes():
        vv = pd.Series(rs[lab == str(m)]).dropna()
        maj[m] = vv.value_counts().index[0] if len(vv) else None
    G = tree.copy()
    for u in [m for m in list(G.nodes()) if maj[m] is None]:
        nb_ = list(G.neighbors(u)); G.add_edges_from((a, b) for i, a in enumerate(nb_) for b in nb_[i + 1:]); G.remove_node(u)
    Sg = nx.Graph(); Sg.add_nodes_from(REF_STATES)
    Sg.add_edges_from((maj[a], maj[b]) for a, b in G.edges() if maj[a] != maj[b])
    return Sg

def edge_f1(Sg):
    E_ref = {frozenset(e) for e in REF_EDGES}; E_pred = {frozenset(e) for e in Sg.edges()}
    tp = len(E_ref & E_pred); p = tp / len(E_pred) if E_pred else 0.0; rc = tp / len(E_ref)
    return 0.0 if tp == 0 else 2 * p * rc / (p + rc)

def score(name, pt, tree=None, lab=None):
    r = {'ordering': name}
    if tree is not None:
        r['IM (↓)'] = im_dist(REF_G, tree); r['OT (↓)'] = ot_dist(REF_G, tree)
        if lab is not None:
            Sg = state_graph(tree, lab)
            r['state-edge F1 (↑)'] = edge_f1(Sg)
            r['state edges found'] = ', '.join(sorted('–'.join(sorted(e)) for e in Sg.edges())) or '-'
    r['KT (↑)'] = kendalltau(pt[ok], rt[ok])[0]; r['SR (↑)'] = spearmanr(pt[ok], rt[ok])[0]
    r['SR beyond distance (↑)'] = partial_sr(pt)
    if lab is not None:
        r['ARI (↑)'] = adjusted_rand_score(rl, lab); r['AMI (↑)'] = adjusted_mutual_info_score(rl, lab)
        r['milestones'] = len(set(lab))
    st_mean = pd.Series(pt[ok]).groupby(rt[ok]).mean()
    r['state order KT (↑)'] = kendalltau(st_mean.index, st_mean.values)[0] if len(st_mean) > 1 else np.nan
    r['stated order recovered'] = 'yes' if np.all(np.diff(st_mean.values) > 0) else 'no'
    return r

rows = [score(nm, T[:, j], trees.get(nm), labs.get(nm)) for j, nm in enumerate(names)]
rows.append(score('SPACECONS (weighted)', borda_w, cons_mtree, CM))
rows.append(score('Borda (uniform)', borda_u))
pc1 = PCA(1).fit_transform(xy)[:, 0] if 'PCA' in globals() else np.linalg.svd(xy - xy.mean(0), full_matrices=False)[0][:, 0]
pc1 = rank01(pc1 if pc1[root] <= np.median(pc1) else -pc1)
rows.append(score('space only (1st PC of x, y)', pc1))
rows.append(score('distance from root (through tissue)', DGEO))

# SPACECONS 2.1 ablation: every weighting rule on the same reference (evaluation only, never used to fit weights).
# The primary rule is the 'SPACECONS (weighted)' row and 'equal' is the 'Borda (uniform)' row, so they are not repeated.
# Do not pick TAU or the rule from these rows: with two datasets that would be fitting to the reference.
for rule in weight_ablation.columns:
    if rule in (WEIGHTING, 'equal'):
        continue
    rows.append(score(f'weighting ablation: {rule}', rank01(T @ weight_ablation[rule].values)))
for col in threshold_weight_sensitivity.columns:
    rows.append(score(f'TAU sensitivity: {col}', rank01(T @ threshold_weight_sensitivity[col].values)))

VAL = pd.DataFrame(rows).set_index('ordering')
print(f'Reference trajectory ({REF_SOURCE}): {", ".join(a + "→" + b for a, b in REF_EDGES)} | cells with a reference state: {int(ok.sum())} of {n}')
display(VAL.round(3))
print('Note: IM and OT compare only the SHAPE of the milestone tree with the reference chain (state names are ignored);'
      f' with {len(REF_STATES)} reference states and many milestones they mostly measure the size difference.'
      ' state-edge F1 is label-aware: 1 = the method links exactly the stated pairs of states '
      f'({", ".join(a + "–" + b for a, b in REF_EDGES)}).')
VAL.round(4).to_csv(f'{OUT}/validation_vs_stlearn.csv')
open(f'{OUT}/spacecons_version.txt', 'w').write('3.1-partial')   # lets the second-dataset cell detect stale folders

st_tab = pd.DataFrame({nm: pd.Series(T[ok, j]).groupby(ad.obs['ref_state'].astype(str).values[ok]).mean() for j, nm in enumerate(names)})
st_tab['SPACECONS'] = pd.Series(borda_w[ok]).groupby(ad.obs['ref_state'].astype(str).values[ok]).mean()
print('Mean pseudotime of each reference state (ordered by reference time; should increase downwards; states with equal time are branches):')
display(st_tab.reindex(REF_STATES).round(3))
st_tab.reindex(REF_STATES).to_csv(f'{OUT}/validation_state_means.csv')
print('Saved validation_vs_stlearn.csv and validation_state_means.csv')


## 10b. SPACECONS-3 — the final consensus (each method does its best job + middle fusion)
| Job | Who contributes | How |
|---|---|---|
| 1. Who may vote | **CASCAT** (causal pruning) + fused neighbour network from **expression, SpaceFlow, CASCAT, stLearn, SpaTrack** | Pairs CASCAT marks as weakest are removed; the rest are weighted by SNF (Wang et al., *Nat Methods* 2014) |
| 2. Ordering | **all five** | Votes on neighbouring pairs, weight = global trust × local gene-trend trust × SpaTrack's confidence; least-squares solve; plus a walker on the fused network; final = mean of the two |
| 3. Structure | **all five** milestones (SPACECONS-M), **StaVia** end points | Consensus milestones/graph; end points checked; Leiden on the fused network as a cross-check |

All settings are fixed in advance (`S3` dict). Reference labels are used only in the validation rows at the end. The ablation rows switch one part
off at a time; use them on TBI/BC (development) only — do not change the recipe after seeing ZHUANG/HER2ST.


In [ ]:
# [CELL:spacecons3]
# =====================  SPACECONS-3: role-based consensus with middle-fusion components  =====================
# Each method does the job it is best at; the methods' internal views are fused where that makes the result more robust.
#   Job 1  who may vote   : spatial 6-NN spot pairs; CASCAT's causal pruning removes 'fake' links; the remaining links are
#                           weighted by a fused neighbour network (SNF over expression, SpaceFlow, CASCAT, stLearn, SpaTrack views)
#   Job 2  ordering       : all five methods vote on every kept pair ('j is later than i by ...'); vote weight =
#                           global trust (leave-one-out agreement²) x local gene-trend trust x method confidence (SpaTrack);
#                           one weighted least-squares solve (root = 0)                      -> estimate A (vote solve)
#                           diffusion pseudotime on the fused network (a 'fused walker')    -> estimate B
#                           final = mean rank of A and B; their agreement is reported per spot
#   Job 3  structure      : consensus milestones + graph from SPACECONS-M; StaVia's terminal states mark end points;
#                           Leiden on the fused network as a cross-check of the milestones
#   Uncertainty           : method bootstrap of the vote solve (B_BOOT draws) + agreement of the two estimates
# Reference labels are NOT used anywhere in this cell except in the validation rows at the very end.
import os, json, glob, time
import numpy as np, pandas as pd, scipy.sparse as sp
from scipy.sparse.linalg import lsqr
from scipy.ndimage import uniform_filter1d
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import adjusted_mutual_info_score

S3 = dict(K_VOTE=6, K_CAND=15, K_SNF=6, SNF_ITERS=20, SNF_MU=0.5, HOOD=18, CASCAT_PERCENT=0.07, B_BOOT=100, PRIOR=0.01)   # fixed in advance
t0_s3 = time.time()
n = ad.n_obs; M = T.shape[1]; meth = list(names)
col_of = {'SpaceFlow': 't_spaceflow', 'SpaTrack': 't_spatrack', 'StaVia': 't_stavia', 'stLearn': 't_stlearn', 'CASCAT': 't_cascat'}
gw = np.asarray(loo_only, float); gw = gw / gw.sum()                       # global trust (leave-one-out agreement, squared)
print('SPACECONS-3 | methods:', meth, '| global trust:', dict(zip(meth, gw.round(3))))

# ---------------------------------------------------------------- graphs on the tissue
_, nb_c = NearestNeighbors(n_neighbors=S3['K_CAND'] + 1).fit(xy).kneighbors(xy)
rc = np.repeat(np.arange(n), S3['K_CAND']); cc = nb_c[:, 1:].ravel()
CAND = sp.csr_matrix((np.ones(len(rc)), (rc, cc)), shape=(n, n)); CAND = ((CAND + CAND.T) > 0).astype(float).tocsr()
_, nb_v = NearestNeighbors(n_neighbors=S3['K_VOTE'] + 1).fit(xy).kneighbors(xy)
E = np.array(sorted({(min(a, b), max(a, b)) for a in range(n) for b in nb_v[a, 1:]})); ei, ej = E[:, 0], E[:, 1]
_, hood = NearestNeighbors(n_neighbors=S3['HOOD'] + 1).fit(xy).kneighbors(xy)
print(f'voting pairs: {len(E)} | candidate pairs for the fused network: {CAND.nnz // 2}')

# ---------------------------------------------------------------- views for the fused network (middle fusion)
def _reindex(arr, barcodes):
    return pd.DataFrame(np.asarray(arr, float), index=pd.Index(barcodes).astype(str)).reindex(ad.obs_names.astype(str)).to_numpy()
views = {'expression': np.asarray(ad.obsm['X_pca'][:, :30], float)}
if os.path.exists(f'{OUT}/spaceflow_embedding.tsv'):
    _e = np.loadtxt(f'{OUT}/spaceflow_embedding.tsv')
    if _e.shape[0] == n: views['SpaceFlow'] = _e
if os.path.exists(f'{OUT}/cascat_result.h5ad'):
    _r = sc.read_h5ad(f'{OUT}/cascat_result.h5ad')
    if 'cascat_embedding' in _r.obsm: views['CASCAT'] = _reindex(_r.obsm['cascat_embedding'], _r.obs_names)
if os.path.exists(f'{OUT}/raw_stlearn_pca.npy') and os.path.exists(f'{OUT}/raw_stlearn.csv'):
    views['stLearn'] = _reindex(np.load(f'{OUT}/raw_stlearn_pca.npy')[:, :30], pd.read_csv(f'{OUT}/raw_stlearn.csv', dtype={'barcode': str})['barcode'])
views = {k: v for k, v in views.items() if np.isfinite(v).all()}

def _affinity_from_embedding(Z):
    """SNF's scaled exponential kernel, evaluated only on candidate (spatially close) pairs."""
    r, c = CAND.nonzero(); d = np.linalg.norm(Z[r] - Z[c], axis=1)
    D = sp.csr_matrix((d, (r, c)), shape=(n, n))
    sig = np.asarray(D.sum(1)).ravel() / np.maximum(np.diff(D.indptr), 1)          # mean distance to the candidate neighbours
    eps = (sig[r] + sig[c] + d) / 3 + 1e-12
    return sp.csr_matrix((np.exp(-d ** 2 / (S3['SNF_MU'] * eps ** 2)), (r, c)), shape=(n, n))
W_views = {k: _affinity_from_embedding(v) for k, v in views.items()}
if os.path.exists(f'{OUT}/spatrack_ot_spatial15.npz'):                              # SpaTrack's transport plan as a similarity
    _ot = sp.load_npz(f'{OUT}/spatrack_ot_spatial15.npz').tocsr(); _ot = (_ot + _ot.T) / 2
    _ot = _ot.multiply(CAND).tocsr()
    if _ot.nnz: W_views['SpaTrack'] = _ot / _ot.data.max()

def _snf(Ws, k=S3['K_SNF'], iters=S3['SNF_ITERS']):
    """Similarity network fusion (Wang et al., Nat Methods 2014), sparse version restricted to spatially close pairs."""
    def full_norm(W):
        W = W.tocsr().astype(float); rs = np.asarray(W.sum(1)).ravel() + 1e-12
        P = sp.diags(0.5 / rs) @ W; return (P + sp.identity(n) * 0.5).tocsr()
    def knn_norm(W):
        W = W.tocsr(); rows, cols, vals = [], [], []
        for i in range(n):
            s, e = W.indptr[i], W.indptr[i + 1]; idx, v = W.indices[s:e], W.data[s:e]
            if len(v) == 0: continue
            top = np.argsort(-v)[:k]; vv = v[top] / (v[top].sum() + 1e-12)
            rows += [i] * len(top); cols += list(idx[top]); vals += list(vv)
        return sp.csr_matrix((vals, (rows, cols)), shape=(n, n))
    mask = (CAND + sp.identity(n)).astype(bool).astype(float)
    P = [full_norm(W) for W in Ws]; S = [knn_norm(W) for W in Ws]
    for _ in range(iters):
        P_new = []
        for v in range(len(Ws)):
            others = sum(P[u] for u in range(len(Ws)) if u != v) / max(1, len(Ws) - 1) if len(Ws) > 1 else P[v]
            Pv = (S[v] @ others @ S[v].T).multiply(mask).tocsr()
            P_new.append(full_norm((Pv + Pv.T) / 2))
        P = P_new
    F = sum(P) / len(P); F = F - sp.diags(F.diagonal()); F = ((F + F.T) / 2).tocsr(); F.eliminate_zeros()
    return F
FUSED = _snf(list(W_views.values()))
print('fused-network views:', list(W_views))

# ---------------------------------------------------------------- Job 1: who may vote
w_fuse = np.asarray(FUSED[ei, ej]).ravel(); w_fuse = np.clip(w_fuse / (np.median(w_fuse[w_fuse > 0]) + 1e-12), 0.1, 3.0)
cut = np.zeros(len(E), bool)
if os.path.exists(f'{OUT}/cascat_CMI.csv'):
    cm_ = pd.read_csv(f'{OUT}/cascat_CMI.csv')
    if {'Cell1', 'Cell2', 'CMI'} <= set(cm_.columns) and cm_[['Cell1', 'Cell2']].max().max() < n:
        g_ = cm_.groupby(['Cell1', 'Cell2'])['CMI'].mean(); g_ = g_[g_ > -np.inf]
        weak = g_[g_ <= g_.quantile(S3['CASCAT_PERCENT'])].index
        weak = {(min(int(a), int(b)), max(int(a), int(b))) for a, b in weak}
        cut = np.array([(int(a), int(b)) in weak for a, b in E])
    else: print('cascat_CMI.csv does not match this dataset -> CASCAT cut not used')
edge_rel = w_fuse * (~cut)
print(f'Job 1: CASCAT cut {int(cut.sum())} of {len(E)} voting pairs; fused-network weight median 1 (range {w_fuse.min():.2f}–{w_fuse.max():.2f})')

# ---------------------------------------------------------------- Job 2: ordering
dT = T[ej] - T[ei]                                                                    # each method's vote on every pair
Xg = np.asarray(ad.obsm['X_pca'][:, :30], float); Xg = (Xg - Xg.mean(0)) / (Xg.std(0) + 1e-12)
win = max(25, n // 40)
def gene_trend_misfit(t):
    """How far each spot's expression is from the typical expression of spots placed at the same time by this order."""
    o = np.argsort(t, kind='stable'); Xs = uniform_filter1d(Xg[o], size=win, axis=0, mode='nearest')
    r = np.empty(n); r[o] = np.linalg.norm(Xg[o] - Xs, axis=1); return r
mis = np.column_stack([gene_trend_misfit(T[:, m]) for m in range(M)])
trust = 1.0 / (1.0 + mis / np.median(mis)); trust = trust[hood].mean(1)              # local gene-trend trust, smoothed
conf = np.ones((n, M))
if 'SpaTrack' in meth and os.path.exists(f'{OUT}/spatrack_confidence.npy'):
    _c = np.load(f'{OUT}/spatrack_confidence.npy')
    if len(_c) == n: conf[:, meth.index('SpaTrack')] = _c

def vote_solve(cols, use_global=True, use_local=True, use_edges=True):
    g_ = gw[cols] / gw[cols].sum() if use_global else np.ones(len(cols)) / len(cols)
    Wsp = (trust[:, cols] if use_local else 1.0) * conf[:, cols] * g_[None, :] + 1e-9
    We = (Wsp[ei] + Wsp[ej]) / 2; We = We / We.sum(1, keepdims=True)
    Fe = (We * dT[:, cols]).sum(1); re = edge_rel if use_edges else np.ones(len(E))
    keep = re > 0; sw = np.sqrt(re[keep])
    B = sp.coo_matrix((np.r_[-sw, sw], (np.r_[np.arange(keep.sum()), np.arange(keep.sum())], np.r_[ei[keep], ej[keep]])),
                      shape=(keep.sum(), n))
    # v3.1f: weak pull of every spot towards the weighted mean of the methods. Without it, a patch cut off from the root by the
    # CASCAT cut has no fixed level and drops to ~0 (TBI: 162 spots). PRIOR is tiny, so connected parts are set by the votes.
    prior = ((Wsp / Wsp.sum(1, keepdims=True)) * T[:, cols]).sum(1); lp = np.sqrt(S3['PRIOR'])
    A_ = sp.vstack([B, sp.coo_matrix(([10.0], ([0], [root])), shape=(1, n)), lp * sp.identity(n, format='csr')]).tocsr()
    tau = lsqr(A_, np.r_[Fe[keep] * sw, 0.0, lp * prior], atol=1e-10, btol=1e-10, iter_lim=20000)[0]
    res = np.abs(B @ tau - Fe[keep] * sw)
    conflict = np.bincount(np.r_[ei[keep], ej[keep]], weights=np.r_[res, res], minlength=n) / \
               np.maximum(np.bincount(np.r_[ei[keep], ej[keep]], minlength=n), 1)
    return rank01(tau), Wsp / Wsp.sum(1, keepdims=True), conflict
allc = list(range(M))
pt_vote, share, conflict = vote_solve(allc)

# estimate B: a 'fused walker' = diffusion pseudotime on the fused network
ew = sc.AnnData(np.zeros((n, 1))); ew.obs_names = ad.obs_names
Cf = FUSED.copy(); Cf.data = Cf.data / Cf.data.max()
Dd = Cf.copy(); Dd.data = 1.0 - Dd.data + 1e-6
ew.obsp['connectivities'] = Cf; ew.obsp['distances'] = Dd
ew.uns['neighbors'] = {'connectivities_key': 'connectivities', 'distances_key': 'distances',
                       'params': {'method': 'umap', 'n_neighbors': S3['K_SNF']}}
try:
    sc.tl.diffmap(ew, n_comps=15); ew.uns['iroot'] = int(root); sc.tl.dpt(ew)
    pt_walk = rank01(ew.obs['dpt_pseudotime'].values)
except Exception as e_:
    print('fused walker failed -> vote solve only:', repr(e_)[:120]); pt_walk = pt_vote.copy()
if pt_walk[(ad.obs['region'] == ROOT_CLUSTER).values].mean() > 0.5: pt_walk = 1 - pt_walk
pt_s3 = rank01((pt_vote + pt_walk) / 2)
agreement = 1 - np.abs(pt_vote - pt_walk)
print(f'Job 2: vote solve vs fused walker Spearman = {spearmanr(pt_vote, pt_walk)[0]:.3f}; median agreement {np.median(agreement):.2f}')

# uncertainty: bootstrap the methods in the vote solve (the fused walker is kept fixed)
rng_s3 = np.random.default_rng(0); boot_s3 = np.empty((S3['B_BOOT'], n))
for b in range(S3['B_BOOT']):
    pick = list(rng_s3.integers(0, M, M)); boot_s3[b] = rank01((vote_solve(pick)[0] + pt_walk) / 2)
lo_s3, hi_s3 = np.percentile(boot_s3, [2.5, 97.5], 0)
print(f'uncertainty: median 95% interval width {np.median(hi_s3 - lo_s3):.3f}')

# ---------------------------------------------------------------- Job 3: structure checks
fused_cl = None
try:
    sc.tl.leiden(ew, adjacency=Cf, resolution=1.0, key_added='fused', flavor='igraph', n_iterations=2, directed=False, random_state=0)
    fused_cl = ew.obs['fused'].astype(str).values
    print(f'Job 3: Leiden on the fused network -> {len(set(fused_cl))} groups; AMI with the consensus milestones = '
          f'{adjusted_mutual_info_score(CM, fused_cl):.3f}')
except Exception as e_: print('fused-network clustering failed:', repr(e_)[:120])
term_ms = []
if os.path.exists(f'{OUT}/stavia_terminal.json') and 'm_stavia' in ad.obs:
    for tcl in json.load(open(f'{OUT}/stavia_terminal.json')).get('terminal_clusters', []):
        mm = pd.Series(np.asarray(CM)[ad.obs['m_stavia'].astype(str).values == str(tcl)]).value_counts()
        if len(mm): term_ms.append(str(mm.index[0]))
    term_ms = sorted(set(term_ms))
    if term_ms:
        _ms_t = pd.Series(pt_s3).groupby(np.asarray(CM)).mean()
        print('StaVia end points fall in consensus milestones:', {m_: round(float(_ms_t[m_]), 3) for m_ in term_ms if m_ in _ms_t},
              '(mean SPACECONS-3 time; should be late)')

# ---------------------------------------------------------------- outputs
most = np.array(meth)[share.argmax(1)]
pd.DataFrame({'barcode': ad.obs_names, 'x': xy[:, 0], 'y': xy[:, 1], 'spacecons3_pseudotime': pt_s3, 'ci_low': lo_s3, 'ci_high': hi_s3,
              'vote_solve': pt_vote, 'fused_walker': pt_walk, 'agreement': agreement, 'conflict': conflict, 'most_trusted': most,
              **{f'share_{m_}': share[:, k] for k, m_ in enumerate(meth)},
              **({'fused_group': fused_cl} if fused_cl is not None else {})}).to_csv(f'{OUT}/spacecons3_per_spot.csv', index=False)
contrib = pd.DataFrame({'global trust': gw, 'mean share': share.mean(0),
                        'most trusted (% spots)': [100 * (most == m_).mean() for m_ in meth]}, index=meth)
contrib.round(3).to_csv(f'{OUT}/spacecons3_method_contribution.csv')
json.dump({'settings': S3, 'fused_views': list(W_views), 'cascat_cut_pairs': int(cut.sum()), 'voting_pairs': int(len(E)),
           'stavia_terminal_milestones': term_ms, 'vote_vs_walker_spearman': float(spearmanr(pt_vote, pt_walk)[0])},
          open(f'{OUT}/spacecons3_summary.json', 'w'), indent=1)
print('What each method contributed:'); display(contrib.round(3))

fig, ax = plt.subplots(2, 3, figsize=(18, 11))
img_s3, pxy_s3 = tissue_image()
panels = [(pt_s3, 'SPACECONS-3 pseudotime', 'viridis'), (pt_vote, 'estimate A: vote solve', 'viridis'),
          (pt_walk, 'estimate B: fused walker', 'viridis'), (agreement, 'agreement of A and B (1 = same)', 'magma'),
          (conflict, 'conflict (votes that cannot be reconciled)', 'magma')]
for a_, (v_, ttl, cmap_) in zip(ax.ravel(), panels):
    show_tissue(a_, .5); s_ = a_.scatter(*pxy_s3.T, c=v_, cmap=cmap_, s=7); plt.colorbar(s_, ax=a_, fraction=.04); a_.set_title(ttl); a_.axis('off')
a_ = ax.ravel()[5]; show_tissue(a_, .5)
for k, m_ in enumerate(meth):
    mk = most == m_; a_.scatter(*pxy_s3[mk].T, s=7, label=f'{m_} ({100 * mk.mean():.0f}%)')
a_.legend(fontsize=9, markerscale=3, loc='lower right'); a_.set_title('most trusted method per spot'); a_.axis('off')
for a_ in ax.ravel()[:5]: a_.scatter(*pxy_s3[root], c='cyan', s=200, marker='*', edgecolors='k')
plt.tight_layout(); plt.savefig(f'{OUT}/fig_SPACECONS3.png', dpi=130); plt.show()

# ---------------------------------------------------------------- validation rows (reference labels used ONLY here)
rows_s3 = [score('SPACECONS-3 (final)', pt_s3, cons_mtree, CM),
           score('SPACECONS-3: vote solve only', pt_vote),
           score('SPACECONS-3: fused walker only', pt_walk),
           score('SPACECONS-3: vote, no CASCAT cut / fused weights', vote_solve(allc, use_edges=False)[0]),
           score('SPACECONS-3: vote, global trust only', vote_solve(allc, use_local=False)[0]),
           score('SPACECONS-3: vote, local trust only', vote_solve(allc, use_global=False)[0])]
VAL = pd.concat([VAL.drop(index=[r['ordering'] for r in rows_s3], errors='ignore'), pd.DataFrame(rows_s3).set_index('ordering')])
VAL.round(4).to_csv(f'{OUT}/validation_vs_stlearn.csv'); open(f'{OUT}/spacecons_version.txt', 'w').write('3.1')
print('Validation of SPACECONS-3 and its ablations (first row = the method; the others switch one part off):')
display(VAL.loc[[r['ordering'] for r in rows_s3] + ['SPACECONS (weighted)', 'Borda (uniform)', 'distance from root (through tissue)'],
                ['SR (↑)', 'KT (↑)', 'SR beyond distance (↑)', 'stated order recovered']].round(3))
print(f'SPACECONS-3 finished in {(time.time() - t0_s3) / 60:.1f} min')


## 11. The other datasets and all datasets side by side
The next cell re-runs every step above for each remaining dataset in `DATASETS` (results in `v3_<DATASET>/`); the cell after it
puts all validation tables together. A dataset already finished with v3.0 is not re-run (delete its folder to redo it).

**Reading the results.** On ZHUANG and HER2ST the labels and the trajectory were curated independently of all five methods, so these
are the first checks of SPACECONS against a reference that is not derived from one of the methods (as BC's was from stLearn).


In [ ]:
# [CELL:second-dataset]
# Runs every step again for the human breast cancer section (results to v3_BC/). A finished dataset is not re-run.
import time
from IPython import get_ipython
ip = get_ipython(); hist = ip.history_manager.input_hist_raw
TAGS = ['download', 'load', 'root', 'spaceflow', 'spatrack', 'stavia', 'stlearn-run', 'cascat-run', 'comparison', 'plots', 'consensus', 'milestones', 'validation', 'spacecons3']
_CELL_SRC = {}
for _TAG in TAGS:
    hits = [h for h in hist if h.lstrip().startswith(f'# [CELL:{_TAG}]')]
    if not hits: raise RuntimeError(f'The "{_TAG}" cell has not run yet in this session: use Run All.')
    _CELL_SRC[_TAG] = hits[-1]
FIRST_OUT = OUT
for NEXT_DATASET in DATASETS[1:]:
    _vfile = f'/kaggle/working/v3_{NEXT_DATASET}/spacecons_version.txt'
    if os.path.exists(_vfile) and open(_vfile).read().strip() == '3.1':
        print(f'{NEXT_DATASET} already done with SPACECONS v3.1, skipped (delete /kaggle/working/v3_{NEXT_DATASET} to redo)'); continue
    for _k in ['t_spaceflow', 't_spatrack', 't_stavia', 't_stlearn', 't_cascat']:   # never carry a method result across datasets
        globals().pop(_k, None)
    for _k in ['H5AD_PATH', 'LABEL_KEY', 'VISIUM_DIR', 'VISIUM_OUT']: globals().pop(_k, None)   # v3.1e: nothing from the previous dataset
    _T0 = time.time()
    for _TAG in TAGS:
        print(f'==================== {NEXT_DATASET}: {_TAG} ====================')
        _r = ip.run_cell(_CELL_SRC[_TAG], store_history=False)
        if isinstance(_r.error_in_exec, KeyboardInterrupt):     # v3.1e: Stop / Cancel run stops the WHOLE loop
            raise KeyboardInterrupt(f'stopped by you during {NEXT_DATASET}: {_TAG}')
        if not _r.success:
            if _TAG in ('download', 'load', 'root'):          # v3.1e: without its data the rest would run on the PREVIOUS dataset
                print(f'!! {_TAG} failed for {NEXT_DATASET}: this dataset is stopped, moving to the next one'); break
            print(f'!! {_TAG} failed for {NEXT_DATASET}; continuing')
    print(f'{NEXT_DATASET} finished in {(time.time() - _T0) / 60:.1f} min')
NEXT_DATASET = None


In [ ]:
# [CELL:summary]
# All datasets side by side: validation against the reference trajectory (CASCAT metrics) + milestone counts
base = '/kaggle/working' if os.path.exists('/kaggle/working') else '.'
tabs, mos = {}, {}
for ds in DATASETS:
    d = f'{base}/v3_{ds}' if os.path.exists(f'{base}/v3_{ds}') else f'outputs_v3_{ds}'
    if os.path.exists(f'{d}/validation_vs_stlearn.csv'): tabs[ds] = pd.read_csv(f'{d}/validation_vs_stlearn.csv', index_col=0)
    if os.path.exists(f'{d}/method_milestone_orderings.csv'): mos[ds] = pd.read_csv(f'{d}/method_milestone_orderings.csv', index_col=0)
if tabs:
    ALL = pd.concat(tabs, axis=1); print('Validation against the reference trajectory, all datasets:'); display(ALL.round(3))
    ALL.round(4).to_csv(f'{base}/validation_all_datasets.csv')
for ds, m in mos.items(): print(f'\n{ds}: number of milestones per method'); display(m[['milestones']].T)
print('Saved validation_all_datasets.csv')


## Root sensitivity (multi-root check)

Candidate roots are taken from the biologically defined root pool and spread over the tissue. SpaceFlow's root-dependent step is re-run from every candidate automatically (`multi_root_spaceflow_pseudotime.csv`); other methods can be re-run separately and saved as `multi_root_method_pseudotime.csv` (columns root_barcode, method, barcode, pseudotime). Only real re-runs are aggregated.


In [ ]:
# [CELL:spacecons20_multiroot]
# Multi-root aggregation for BOTH datasets (2.0 ran it only for the dataset still in memory, i.e. BC).
# Inputs per dataset folder: multi_root_candidates.csv (root cell) and every multi_root_*_pseudotime.csv:
#   multi_root_spaceflow_pseudotime.csv  - made automatically by the SpaceFlow cell (real reruns)
#   multi_root_method_pseudotime.csv     - optional, your reruns of other methods; columns root_barcode, method, barcode, pseudotime
import glob, itertools, os
import numpy as np, pandas as pd
from scipy.stats import spearmanr


def _orient_rank(g, rb):
    vals = g['pseudotime'].to_numpy(float)
    r = (pd.Series(vals).rank(method='average').to_numpy() - 1) / max(1, len(vals) - 1)
    at_root = r[(g['barcode'].astype(str) == str(rb)).to_numpy()]
    if len(at_root) == 0:
        print(f'  warning: root {rb} missing from {g["method"].iloc[0]} output; orientation not checked')
    elif at_root.mean() > 0.5:
        r = 1 - r
    return r


def aggregate_multiroot_runs(run_table):
    """Rank-normalise and orient each (root, method) run; average methods within a root, then roots equally.
    Returns per-spot summary, per-root consensus and a per-method root-sensitivity table."""
    need = {'root_barcode', 'method', 'barcode', 'pseudotime'}
    if need - set(run_table.columns):
        raise ValueError(f'Missing columns: {sorted(need - set(run_table.columns))}')
    q = run_table.assign(pseudotime=pd.to_numeric(run_table['pseudotime'], errors='coerce'))
    q = q[np.isfinite(q['pseudotime'])]
    parts = []
    for (rb, m), g in q.groupby(['root_barcode', 'method'], sort=False):
        if len(g) > 1:
            parts.append(g[['root_barcode', 'method', 'barcode']].assign(rank_pseudotime=_orient_rank(g, rb)))
    z = pd.concat(parts, ignore_index=True)
    per_root = z.groupby(['root_barcode', 'barcode'])['rank_pseudotime'].mean().rename('root_consensus').reset_index()
    overall = (per_root.groupby('barcode')['root_consensus'].agg(['mean', 'std', 'count']).reset_index()
               .rename(columns={'mean': 'multiroot_pseudotime', 'std': 'root_sd', 'count': 'n_roots'}))
    overall['root_sd'] = overall['root_sd'].fillna(0)
    sens = []
    for m, g in z.groupby('method'):
        P = g.pivot_table(index='barcode', columns='root_barcode', values='rank_pseudotime')
        rhos = [spearmanr(P[a], P[b], nan_policy='omit')[0] for a, b in itertools.combinations(P.columns, 2)]
        sens.append({'method': m, 'roots': P.shape[1], 'median rho between roots': np.median(rhos) if rhos else np.nan,
                     'min rho between roots': np.min(rhos) if rhos else np.nan})
    return overall, per_root, pd.DataFrame(sens)


_base = '/kaggle/working' if os.path.exists('/kaggle/working') else '.'
for _ds in DATASETS:
    _d = f'{_base}/v3_{_ds}'
    if not os.path.exists(f'{_d}/multi_root_candidates.csv'):
        print(f'{_ds}: no candidate roots (run the root cell for this dataset)'); continue
    _runs = [pd.read_csv(f) for f in sorted(glob.glob(f'{_d}/multi_root_*_pseudotime.csv'))]
    if not _runs:
        print(f'{_ds}: candidate roots exported; no per-root reruns found yet'); continue
    _summ, _byroot, _sens = aggregate_multiroot_runs(pd.concat(_runs, ignore_index=True))
    _summ.to_csv(f'{_d}/multiroot_consensus_per_spot.csv', index=False)
    _byroot.to_csv(f'{_d}/multiroot_consensus_by_root.csv', index=False)
    _sens.to_csv(f'{_d}/root_sensitivity.csv', index=False)
    print(f'{_ds}: methods with reruns = {sorted(_sens.method)}; median root SD = {_summ.root_sd.median():.3f}')
    display(_sens.round(3))
print('Only methods that were actually re-run are included; no other method is relabelled as a multi-root result.')


## 9. Download the results
Run this cell and click the links (or use the Output panel). Better: **Save Version → Save & Run All (Commit)**.

In [ ]:
from IPython.display import FileLink, display
for f in sorted(glob.glob(f'{OUT}/*.csv')):
    display(FileLink(os.path.relpath(f)))

In [ ]:
import os, shutil, glob
from IPython.display import FileLink, display
for d in sorted(glob.glob('/kaggle/working/v3_*')):
    if os.path.isdir(d):
        z = shutil.make_archive(d, 'zip', d)
        print(os.path.basename(z), f'{os.path.getsize(z) / 1e6:.1f} MB')
        display(FileLink(os.path.relpath(z, '/kaggle/working')))

### Reproducibility notes

- Reference labels are used only in the validation cells, never to build an ordering, milestones or graph.
- All SPACECONS-3 settings (`S3` dict) were fixed on TBI and BC before ZHUANG and HER2ST were run (held-out test).
- SpaceFlow, StaVia and CASCAT use the GPU and random initialisation, so re-runs differ slightly (second decimal).
- A finished dataset is skipped by the dataset loop; delete `/kaggle/working/v3_<DATASET>/spacecons_version.txt` to redo it.
- If figshare refuses downloads from Kaggle (HTTP 403), download files 52119332, 52119302 and 52119398 in a browser and add them as a Kaggle dataset (+ Add Input); they are found automatically by size.
